In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/Lung_Colon/WCE/ /content/

In [ ]:
!cp -r /content/drive/MyDrive/ViT /content/

In [ ]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00


In [ ]:
!pip install -q timm transformers nibabel open_clip_torch segment-anything huggingface_hub

In [ ]:
# ===================================== 2D IMAGE RETRIEVAL BENCHMARK =====================================
# Folder structure:
#   dataset_root/
#       class_a/
#           image1.jpg
#           image2.png
#       class_b/
#           image3.jpg
#           ...
#
# What it does:
#   - Stratified 80/20 split by class
#   - Extract embeddings for both train and test splits
#   - Retrieval: for each test image, retrieve top-k most similar train images
#   - Evaluate whether retrieved neighbors have the same class as the query
#
# Metrics:
#   - Precision@K
#   - HitRate@K
#   - Recall@K
#   - mAP
#   - MRR
#
# Added models:
#   - MedSigLIP
#   - SigLIP
#   - UNI (medical foundation model; requires access if gated)
#   - OpenCLIP ViT-H/14
# ========================================================================================================

import os
import gc
import random
from pathlib import Path
from typing import List, Dict, Optional

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Install if needed:
# !pip install -q timm transformers open_clip_torch segment-anything huggingface_hub pillow scikit-learn

import timm
from sklearn.model_selection import train_test_split
from transformers import CLIPVisionModel, SamModel, AutoModel
from huggingface_hub import hf_hub_download

try:
    import open_clip
    HAS_OPEN_CLIP = True
except Exception:
    HAS_OPEN_CLIP = False

try:
    from segment_anything import sam_model_registry
    HAS_SEGMENT_ANYTHING = True
except Exception:
    HAS_SEGMENT_ANYTHING = False


# ===================================== USER CONFIG =====================================
DATASET_ROOT = "/content/WCE"
RESULTS_DIR = "./retrieval_outputs_2d"
os.makedirs(RESULTS_DIR, exist_ok=True)

BACKBONES_TO_TEST = [
    "resnet50",
    "vit_b16",
    "swin_b",
    "clip_vit_l14",
    "biomedclip",
    "dinov2_vitb14",
    "mae_vit_b16",
    "bmc_clip_cf",
    "sam_vit_b",
    "medsam_vit_b",
    "siglip",
    "medsiglip",
    "uni",
    "openclip_vit_h14",
]

MODEL_PATHS = {
    "clip_vit_l14": "openai/clip-vit-large-patch14",
    "biomedclip": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    "sam_vit_b": "facebook/sam-vit-base",

    # Put your local MedSAM checkpoint here
    "medsam_vit_b_ckpt": "/content/ViT/medsam_vit_b.pth",

    # BMC-CLIP-CF
    "bmc_clip_cf_repo": "BIOMEDICA/BMC_CLIP_CF",
    "bmc_clip_cf_file": "BMC_CLIP_CF.pt",
    "bmc_clip_cf_base_model": "ViT-L-14",
    "bmc_clip_cf_base_pretrained": "commonpool_xl_clip_s13b_b90k",
    "bmc_clip_cf_alpha": 0.0,

    # Added
    "siglip": "google/siglip-base-patch16-224",
    "medsiglip": "google/medsiglip-448",

    # UNI can be gated; ensure access if needed
    "uni": "hf-hub:MahmoodLab/UNI",

    # OpenCLIP ViT-H/14
    "openclip_vit_h14_model": "ViT-H-14",
    "openclip_vit_h14_pretrained": "laion2b_s32b_b79k",
}

IMAGE_SIZE = 224
TOP_KS = (1, 5, 10)
BATCH_SIZE = 32
NUM_WORKERS = 2
EMBED_DIM = 512
USE_BFLOAT16 = True
RANDOM_SEED = 42
SHOW_RESULTS = True
NUM_EXAMPLE_QUERIES = 5

ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
# =======================================================================================


# ===================================== NORMALIZATION =====================================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

CLIP_MEAN = torch.tensor([0.48145466, 0.45782750, 0.40821073], dtype=torch.float32).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], dtype=torch.float32).view(1, 3, 1, 1)


def imagenet_norm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(device=x.device, dtype=x.dtype)
    std = IMAGENET_STD.to(device=x.device, dtype=x.dtype)
    return (x - mean) / std


def clip_norm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = CLIP_MEAN.to(device=x.device, dtype=x.dtype)
    std = CLIP_STD.to(device=x.device, dtype=x.dtype)
    return (x - mean) / std


def imagenet_unnorm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(device=x.device, dtype=x.dtype)
    std = IMAGENET_STD.to(device=x.device, dtype=x.dtype)
    return x * std + mean


# ===================================== UTILS =====================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_open_rgb(path: str) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        raise RuntimeError(f"Could not open image {path}: {e}")


def list_images_by_class(root: str) -> pd.DataFrame:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {root}")

    rows = []
    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])

    if len(class_dirs) == 0:
        raise ValueError(f"No class folders found in {root}")

    for class_dir in class_dirs:
        class_name = class_dir.name
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in ALLOWED_EXTS:
                rows.append({
                    "image_path": str(p),
                    "class_name": class_name,
                })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise ValueError(f"No images found in {root}")

    classes = sorted(df["class_name"].unique().tolist())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    df["label"] = df["class_name"].map(class_to_idx)
    return df


def stratified_split(df: pd.DataFrame, test_size: float = 0.2, seed: int = 42):
    counts = df["label"].value_counts()
    valid_labels = counts[counts >= 2].index.tolist()
    dropped = df[~df["label"].isin(valid_labels)].copy()
    df = df[df["label"].isin(valid_labels)].copy().reset_index(drop=True)

    if len(dropped) > 0:
        print(f"[WARN] Dropped {len(dropped)} images from classes with <2 images")

    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"].values,
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def average_precision_from_flags(relevant_flags: List[int]) -> float:
    relevant_flags = np.asarray(relevant_flags, dtype=np.int32)
    total_rel = relevant_flags.sum()
    if total_rel == 0:
        return 0.0

    precisions = []
    hits = 0
    for i, rel in enumerate(relevant_flags, start=1):
        if rel:
            hits += 1
            precisions.append(hits / i)
    return float(np.mean(precisions)) if precisions else 0.0


def reciprocal_rank(relevant_flags: List[int]) -> float:
    for i, rel in enumerate(relevant_flags, start=1):
        if rel:
            return 1.0 / i
    return 0.0


# ===================================== DATASET =====================================
class ImageFolderRetrievalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def _preprocess(self, img: Image.Image) -> torch.Tensor:
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        x = np.asarray(img, dtype=np.float32) / 255.0
        x = np.transpose(x, (2, 0, 1))  # HWC -> CHW
        x = torch.from_numpy(x)

        # Store dataset output as ImageNet-normalized, then convert inside retriever
        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
        x = (x - mean) / std
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_rgb(row["image_path"])
        x = self._preprocess(img)
        return {
            "image": x,
            "label": int(row["label"]),
            "class_name": row["class_name"],
            "image_path": row["image_path"],
        }


# ===================================== BMC-CLIP-CF LOADER =====================================
def load_bmc_clip_cf():
    if not HAS_OPEN_CLIP:
        raise ImportError("BMC_CLIP_CF requires open_clip_torch.")

    repo_id = MODEL_PATHS["bmc_clip_cf_repo"]
    filename = MODEL_PATHS["bmc_clip_cf_file"]
    model_name = MODEL_PATHS["bmc_clip_cf_base_model"]
    pretrained = MODEL_PATHS["bmc_clip_cf_base_pretrained"]
    alpha = float(MODEL_PATHS.get("bmc_clip_cf_alpha", 0.0))

    ckpt_path = hf_hub_download(repo_id=repo_id, filename=filename)
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint["state_dict"]
    del checkpoint

    model, _, _ = open_clip.create_model_and_transforms(
        model_name=model_name,
        pretrained=pretrained,
    )

    model_state_dict = model.state_dict()

    print(f"Merging BMC-CLIP-CF with alpha={alpha}")
    if alpha == 0:
        print("Using only BMC-CLIP-CF checkpoint weights")

    for key in model_state_dict.keys():
        ckpt_key = f"module.{key}"
        if ckpt_key in state_dict:
            model_state_dict[key] = alpha * model_state_dict[key] + (1 - alpha) * state_dict[ckpt_key]

    model.load_state_dict(model_state_dict, strict=True)
    return model


# ===================================== FEATURE EXTRACTORS =====================================
def create_feature_extractor(backbone_name: str):
    """
    Returns:
      model, feat_dim, encoder_kind, preferred_image_size
    """
    backbone_name = backbone_name.lower()

    if backbone_name == "resnet50":
        m = timm.create_model("resnet50", pretrained=True, num_classes=0)
        return m, m.num_features, "timm_slice2d", 224

    elif backbone_name == "vit_b16":
        m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
        return m, m.embed_dim, "timm_slice2d", 224

    elif backbone_name == "swin_b":
        m = timm.create_model("swin_base_patch4_window7_224", pretrained=True, num_classes=0)
        return m, m.num_features, "timm_slice2d", 224

    elif backbone_name == "clip_vit_l14":
        m = CLIPVisionModel.from_pretrained(MODEL_PATHS["clip_vit_l14"])
        return m, m.config.hidden_size, "clip_slice2d", 224

    elif backbone_name == "biomedclip":
        if not HAS_OPEN_CLIP:
            raise ImportError("BiomedCLIP requires open_clip_torch.")
        m, _, _ = open_clip.create_model_and_transforms(MODEL_PATHS["biomedclip"])
        feat_dim = getattr(m.visual, "output_dim", 512)
        return m, feat_dim, "openclip_slice2d", 224

    elif backbone_name == "dinov2_vitb14":
        m = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        return m, m.embed_dim, "dinov2_slice2d", 224

    elif backbone_name == "mae_vit_b16":
        m = timm.create_model("vit_base_patch16_224.mae", pretrained=True, num_classes=0)
        return m, m.embed_dim, "timm_slice2d", 224

    elif backbone_name == "bmc_clip_cf":
        m = load_bmc_clip_cf()
        feat_dim = getattr(m.visual, "output_dim", 768)
        return m, feat_dim, "openclip_slice2d", 224

    elif backbone_name == "sam_vit_b":
        m = SamModel.from_pretrained(MODEL_PATHS["sam_vit_b"])
        feat_dim = int(getattr(m.config.vision_config, "output_channels", 256))
        return m, feat_dim, "sam_hf_slice2d", 1024

    elif backbone_name == "medsam_vit_b":
        if not HAS_SEGMENT_ANYTHING:
            raise ImportError("MedSAM requires segment-anything.")
        ckpt = MODEL_PATHS["medsam_vit_b_ckpt"]
        if not os.path.isfile(ckpt):
            raise FileNotFoundError(f"MedSAM checkpoint not found: {ckpt}")
        m = sam_model_registry["vit_b"](checkpoint=ckpt)
        return m, 256, "sam_repo_slice2d", 512

    elif backbone_name == "siglip":
        m = AutoModel.from_pretrained(MODEL_PATHS["siglip"])
        hidden = getattr(getattr(m, "config", None), "hidden_size", None)
        if hidden is None and hasattr(m, "vision_model") and hasattr(m.vision_model.config, "hidden_size"):
            hidden = m.vision_model.config.hidden_size
        if hidden is None:
            hidden = 768
        return m, hidden, "siglip_slice2d", 224

    elif backbone_name == "medsiglip":
        m = AutoModel.from_pretrained(MODEL_PATHS["medsiglip"])
        hidden = getattr(getattr(m, "config", None), "hidden_size", None)
        if hidden is None and hasattr(m, "vision_model") and hasattr(m.vision_model.config, "hidden_size"):
            hidden = m.vision_model.config.hidden_size
        if hidden is None:
            hidden = 768
        return m, hidden, "medsiglip_slice2d", 448

    elif backbone_name == "uni":
        try:
            m = timm.create_model(MODEL_PATHS["uni"], pretrained=True, num_classes=0)
        except Exception:
            # Some releases/cards recommend these extra args
            m = timm.create_model(
                MODEL_PATHS["uni"],
                pretrained=True,
                num_classes=0,
                init_values=1e-5,
                dynamic_img_size=True,
            )

        feat_dim = getattr(m, "num_features", None)
        if feat_dim is None:
            feat_dim = getattr(m, "embed_dim", 1024)
        return m, feat_dim, "uni_slice2d", 224

    elif backbone_name == "openclip_vit_h14":
        if not HAS_OPEN_CLIP:
            raise ImportError("openclip_vit_h14 requires open_clip_torch.")
        m, _, _ = open_clip.create_model_and_transforms(
            model_name=MODEL_PATHS["openclip_vit_h14_model"],
            pretrained=MODEL_PATHS["openclip_vit_h14_pretrained"],
        )
        feat_dim = getattr(m.visual, "output_dim", 1024)
        return m, feat_dim, "openclip_slice2d", 224

    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")


# ===================================== RETRIEVER =====================================
class UniversalImageRetriever(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        embed_dim: Optional[int] = 512,
        use_bfloat16: bool = True,
        image_size: int = 224,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.encoder, self.hidden_size, self.encoder_kind, self.preferred_image_size = create_feature_extractor(self.backbone_name)

        self.image_size = self.preferred_image_size if self.preferred_image_size is not None else image_size

        self.proj = None
        self.output_dim = self.hidden_size
        if embed_dim is not None and embed_dim != self.hidden_size:
            self.proj = nn.Linear(self.hidden_size, embed_dim)
            self.output_dim = embed_dim

        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()

        safe_bf16_kinds = {
            "timm_slice2d",
            "clip_slice2d",
            "openclip_slice2d",
            "dinov2_slice2d",
            "uni_slice2d",
            "siglip_slice2d",
            "medsiglip_slice2d",
        }
        if use_bfloat16 and torch.cuda.is_available() and self.encoder_kind in safe_bf16_kinds:
            self.encoder = self.encoder.to(dtype=torch.bfloat16)

    def _adapt_input_for_backbone(self, x: torch.Tensor) -> torch.Tensor:
        """
        Dataset outputs ImageNet-normalized tensors.
        Convert back to [0,1], then apply the right normalization per backbone.
        """
        x = imagenet_unnorm_batch(x.float()).clamp(0.0, 1.0)

        if self.encoder_kind in {"clip_slice2d", "openclip_slice2d", "siglip_slice2d", "medsiglip_slice2d"}:
            x = clip_norm_batch(x)
        elif self.encoder_kind in {"timm_slice2d", "dinov2_slice2d", "sam_hf_slice2d", "sam_repo_slice2d", "uni_slice2d"}:
            x = imagenet_norm_batch(x)
        else:
            x = imagenet_norm_batch(x)

        return x

    def _normalize_encoder_output(self, feats):
        if isinstance(feats, dict):
            for key in [
                "image_embeds",
                "x_norm_clstoken",
                "x_cls_token",
                "cls_token",
                "pooler_output",
                "last_hidden_state",
                "x_norm_patchtokens",
            ]:
                if key in feats:
                    feats = feats[key]
                    break
            else:
                first_key = list(feats.keys())[0]
                feats = feats[first_key]

        if isinstance(feats, (list, tuple)):
            feats = feats[0]

        if hasattr(feats, "last_hidden_state"):
            feats = feats.last_hidden_state

        if not torch.is_tensor(feats):
            raise TypeError(f"Unsupported feature output type: {type(feats)}")

        if feats.ndim == 4:
            feats = feats.mean(dim=(2, 3))
        elif feats.ndim == 3:
            feats = feats.mean(dim=1)
        elif feats.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected feature tensor shape: {tuple(feats.shape)}")

        return feats

    def _forward_in_chunks(self, x: torch.Tensor, fn, chunk_size: int = 2) -> torch.Tensor:
        outs = []
        for i in range(0, x.shape[0], chunk_size):
            xi = x[i:i + chunk_size]
            yi = fn(xi)
            outs.append(yi)
        return torch.cat(outs, dim=0)

    def _encode_2d_batch(self, x2d: torch.Tensor) -> torch.Tensor:
        if self.encoder_kind == "timm_slice2d":
            if self.backbone_name == "swin_b":
                feats = self.encoder.forward_features(x2d)

                if isinstance(feats, (list, tuple)):
                    feats = feats[0]

                if not torch.is_tensor(feats):
                    raise TypeError(f"Unsupported Swin output type: {type(feats)}")

                if feats.ndim == 4:
                    if feats.shape[-1] == self.hidden_size:
                        feats = feats.mean(dim=(1, 2))   # NHWC
                    elif feats.shape[1] == self.hidden_size:
                        feats = feats.mean(dim=(2, 3))   # NCHW
                    else:
                        raise ValueError(f"Unexpected Swin feature shape: {tuple(feats.shape)}")
                elif feats.ndim == 3:
                    feats = feats.mean(dim=1)
                elif feats.ndim == 2:
                    pass
                else:
                    raise ValueError(f"Unexpected Swin feature shape: {tuple(feats.shape)}")

                return feats

            else:
                if hasattr(self.encoder, "forward_features"):
                    feats = self.encoder.forward_features(x2d)
                else:
                    feats = self.encoder(x2d)
                feats = self._normalize_encoder_output(feats)
                return feats

        elif self.encoder_kind == "clip_slice2d":
            out = self.encoder(pixel_values=x2d)
            feats = out.last_hidden_state.mean(dim=1)
            return feats

        elif self.encoder_kind == "openclip_slice2d":
            if hasattr(self.encoder, "encode_image"):
                feats = self.encoder.encode_image(x2d)
            else:
                feats = self.encoder.visual(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "dinov2_slice2d":
            if hasattr(self.encoder, "forward_features"):
                feats = self.encoder.forward_features(x2d)
            else:
                feats = self.encoder(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "sam_hf_slice2d":
            x = x2d.float()
            if x.max() <= 1.0:
                x = x * 255.0

            feats = self._forward_in_chunks(
                x,
                lambda z: self.encoder.get_image_embeddings(pixel_values=z),
                chunk_size=1,
            )
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "sam_repo_slice2d":
            x = x2d.float()
            if x.max() <= 1.0:
                x = x * 255.0

            def run_repo_sam(z):
                zz = self.encoder.preprocess(z) if hasattr(self.encoder, "preprocess") else z
                return self.encoder.image_encoder(zz)

            feats = self._forward_in_chunks(x, run_repo_sam, chunk_size=1)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "siglip_slice2d":
            # Prefer direct image feature API if exposed
            if hasattr(self.encoder, "get_image_features"):
                feats = self.encoder.get_image_features(pixel_values=x2d)
            else:
                out = self.encoder(pixel_values=x2d)
                if hasattr(out, "image_embeds") and out.image_embeds is not None:
                    feats = out.image_embeds
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state.mean(dim=1)
                else:
                    feats = self._normalize_encoder_output(out)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "medsiglip_slice2d":
            if hasattr(self.encoder, "get_image_features"):
                feats = self.encoder.get_image_features(pixel_values=x2d)
            else:
                out = self.encoder(pixel_values=x2d)
                if hasattr(out, "image_embeds") and out.image_embeds is not None:
                    feats = out.image_embeds
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state.mean(dim=1)
                else:
                    feats = self._normalize_encoder_output(out)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "uni_slice2d":
            if hasattr(self.encoder, "forward_features"):
                feats = self.encoder.forward_features(x2d)
            else:
                feats = self.encoder(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        else:
            raise ValueError(f"Unknown encoder kind: {self.encoder_kind}")

    @torch.inference_mode()
    def encode_images(self, x: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device
        encoder_params = list(self.encoder.parameters())
        enc_dtype = encoder_params[0].dtype if len(encoder_params) > 0 else torch.float32

        # Input comes ImageNet-normalized from dataset -> adapt for backbone
        x = x.to(device=device, dtype=torch.float32)
        x = self._adapt_input_for_backbone(x)

        # Resize if needed for backbone
        if x.shape[-1] != self.image_size or x.shape[-2] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size), mode="bilinear", align_corners=False)

        x = x.to(device=device, dtype=enc_dtype)
        feats = self._encode_2d_batch(x)

        if self.proj is not None:
            feats = self.proj(feats.float())

        feats = F.normalize(feats.float(), p=2, dim=-1)
        return feats


# ===================================== EMBEDDINGS =====================================
def build_embeddings(
    df: pd.DataFrame,
    retriever: UniversalImageRetriever,
    split_name: str,
    batch_size: int = 32,
    num_workers: int = 2,
    image_size: int = 224,
):
    dataset = ImageFolderRetrievalDataset(df=df, image_size=image_size)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    rows = []
    embeds = []

    retriever.eval()

    for i, batch in enumerate(loader, start=1):
        x = batch["image"]
        labels = batch["label"].numpy()
        class_names = batch["class_name"]
        image_paths = batch["image_path"]

        feats = retriever.encode_images(x).detach().cpu().numpy().astype(np.float32)
        embeds.append(feats)

        for j in range(len(image_paths)):
            rows.append({
                "image_path": image_paths[j],
                "label": int(labels[j]),
                "class_name": class_names[j],
                "split": split_name,
            })

        if i % 10 == 0 or i == len(loader):
            print(f"[{split_name}] Processed {i}/{len(loader)} batches")

    embeddings = np.vstack(embeds).astype(np.float32)
    index_df = pd.DataFrame(rows)
    return index_df, embeddings


# ===================================== RETRIEVAL =====================================
def retrieve_topk(
    query_embeddings: np.ndarray,
    gallery_embeddings: np.ndarray,
    top_k: int = 5,
):
    query_embeddings = query_embeddings / (np.linalg.norm(query_embeddings, axis=1, keepdims=True) + 1e-8)
    gallery_embeddings = gallery_embeddings / (np.linalg.norm(gallery_embeddings, axis=1, keepdims=True) + 1e-8)

    sims = query_embeddings @ gallery_embeddings.T
    topk_idx = np.argsort(sims, axis=1)[:, ::-1][:, :top_k]
    topk_sims = np.take_along_axis(sims, topk_idx, axis=1)
    return topk_idx, topk_sims


def evaluate_retrieval(
    query_df: pd.DataFrame,
    query_embeddings: np.ndarray,
    gallery_df: pd.DataFrame,
    gallery_embeddings: np.ndarray,
    ks=(1, 5, 10),
):
    query_labels = query_df["label"].values
    gallery_labels = gallery_df["label"].values

    max_k = max(ks)
    topk_idx, topk_sims = retrieve_topk(
        query_embeddings=query_embeddings,
        gallery_embeddings=gallery_embeddings,
        top_k=max_k,
    )

    precision_at_k = {k: [] for k in ks}
    hitrate_at_k = {k: [] for k in ks}
    recall_at_k = {k: [] for k in ks}
    ap_list = []
    rr_list = []
    examples = []

    gallery_label_counts = pd.Series(gallery_labels).value_counts().to_dict()

    for q_idx in range(len(query_df)):
        q_label = int(query_labels[q_idx])
        q_class = query_df.iloc[q_idx]["class_name"]
        q_path = query_df.iloc[q_idx]["image_path"]

        retrieved_indices = topk_idx[q_idx]
        retrieved_sims = topk_sims[q_idx]
        retrieved_labels = gallery_labels[retrieved_indices]
        relevant_flags = (retrieved_labels == q_label).astype(np.int32).tolist()

        total_relevant_in_gallery = int(gallery_label_counts.get(q_label, 0))

        for k in ks:
            flags_k = relevant_flags[:k]
            num_rel_k = sum(flags_k)

            precision_at_k[k].append(num_rel_k / k)
            hitrate_at_k[k].append(1.0 if num_rel_k > 0 else 0.0)

            if total_relevant_in_gallery > 0:
                recall_at_k[k].append(num_rel_k / total_relevant_in_gallery)
            else:
                recall_at_k[k].append(0.0)

        ap_list.append(average_precision_from_flags(relevant_flags))
        rr_list.append(reciprocal_rank(relevant_flags))

        ex = {
            "query_path": q_path,
            "query_class": q_class,
            "retrieved": []
        }
        for rank, (g_idx, sim) in enumerate(zip(retrieved_indices, retrieved_sims), start=1):
            ex["retrieved"].append({
                "rank": rank,
                "image_path": gallery_df.iloc[g_idx]["image_path"],
                "class_name": gallery_df.iloc[g_idx]["class_name"],
                "label": int(gallery_df.iloc[g_idx]["label"]),
                "similarity": float(sim),
                "correct_class": bool(gallery_df.iloc[g_idx]["label"] == q_label),
            })
        examples.append(ex)

    metrics = {
        "num_queries": int(len(query_df)),
        "gallery_size": int(len(gallery_df)),
        "mAP": float(np.mean(ap_list)) if len(ap_list) > 0 else 0.0,
        "MRR": float(np.mean(rr_list)) if len(rr_list) > 0 else 0.0,
    }

    for k in ks:
        metrics[f"Precision@{k}"] = float(np.mean(precision_at_k[k])) if len(precision_at_k[k]) > 0 else 0.0
        metrics[f"HitRate@{k}"] = float(np.mean(hitrate_at_k[k])) if len(hitrate_at_k[k]) > 0 else 0.0
        metrics[f"Recall@{k}"] = float(np.mean(recall_at_k[k])) if len(recall_at_k[k]) > 0 else 0.0

    return metrics, examples


# ===================================== VIS =====================================
def show_example_retrievals(examples: List[Dict], num_examples: int = 5):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("matplotlib not available, skipping visualization.")
        return

    chosen = examples[:num_examples]
    if len(chosen) == 0:
        return

    for ex in chosen:
        retrieved = ex["retrieved"]
        ncols = len(retrieved) + 1

        plt.figure(figsize=(3 * ncols, 3))

        q_img = safe_open_rgb(ex["query_path"])
        ax = plt.subplot(1, ncols, 1)
        ax.imshow(q_img)
        ax.set_title(f"QUERY\n{ex['query_class']}")
        ax.axis("off")

        for i, item in enumerate(retrieved, start=2):
            img = safe_open_rgb(item["image_path"])
            ax = plt.subplot(1, ncols, i)
            ax.imshow(img)
            ax.set_title(
                f"R{item['rank']}\n{item['class_name']}\nsim={item['similarity']:.3f}\n"
                f"{'OK' if item['correct_class'] else 'WRONG'}",
                fontsize=9
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()


# ===================================== SAVE =====================================
def save_split_and_embeddings(split_df: pd.DataFrame, embeddings: np.ndarray, prefix: str, out_dir: str):
    csv_path = os.path.join(out_dir, f"{prefix}_index.csv")
    npy_path = os.path.join(out_dir, f"{prefix}_embeddings.npy")

    split_df.to_csv(csv_path, index=False)
    np.save(npy_path, embeddings)

    print(f"Saved {prefix} CSV: {csv_path}")
    print(f"Saved {prefix} embeddings: {npy_path}")
    print(f"{prefix} embedding shape: {embeddings.shape}")


# ===================================== MAIN =====================================
def main():
    set_seed(RANDOM_SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    df = list_images_by_class(DATASET_ROOT)
    print(f"Total images found: {len(df)}")
    print(f"Total classes found: {df['label'].nunique()}")
    print("\nClass distribution:")
    print(df["class_name"].value_counts())

    train_df, test_df = stratified_split(df, test_size=0.2, seed=RANDOM_SEED)
    print(f"\nTrain size: {len(train_df)}")
    print(f"Test size:  {len(test_df)}")

    all_metrics = []

    for backbone_name in BACKBONES_TO_TEST:
        print("\n" + "=" * 100)
        print(f"BACKBONE: {backbone_name}")
        print("=" * 100)

        try:
            retriever = UniversalImageRetriever(
                backbone_name=backbone_name,
                embed_dim=EMBED_DIM,
                use_bfloat16=USE_BFLOAT16,
                image_size=IMAGE_SIZE,
            ).to(device).eval()

            # Dataset stays simple at IMAGE_SIZE; retriever resizes internally for each backbone
            train_index_df, train_embeddings = build_embeddings(
                df=train_df,
                retriever=retriever,
                split_name="train",
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                image_size=IMAGE_SIZE,
            )

            test_index_df, test_embeddings = build_embeddings(
                df=test_df,
                retriever=retriever,
                split_name="test",
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                image_size=IMAGE_SIZE,
            )

            safe_name = backbone_name.replace("/", "_")
            out_dir = os.path.join(RESULTS_DIR, safe_name)
            os.makedirs(out_dir, exist_ok=True)

            save_split_and_embeddings(train_index_df, train_embeddings, "train", out_dir)
            save_split_and_embeddings(test_index_df, test_embeddings, "test", out_dir)

            metrics, examples = evaluate_retrieval(
                query_df=test_index_df,
                query_embeddings=test_embeddings,
                gallery_df=train_index_df,
                gallery_embeddings=train_embeddings,
                ks=TOP_KS,
            )

            metrics["backbone"] = backbone_name
            all_metrics.append(metrics)

            print("\nMetrics:")
            print(pd.DataFrame([metrics]).T)

            rows = []
            for ex in examples:
                for item in ex["retrieved"]:
                    rows.append({
                        "query_path": ex["query_path"],
                        "query_class": ex["query_class"],
                        "rank": item["rank"],
                        "retrieved_path": item["image_path"],
                        "retrieved_class": item["class_name"],
                        "similarity": item["similarity"],
                        "correct_class": item["correct_class"],
                    })
            pd.DataFrame(rows).to_csv(os.path.join(out_dir, "retrieval_examples.csv"), index=False)

            if SHOW_RESULTS:
                print(f"\nShowing example retrievals for {backbone_name}")
                show_example_retrievals(examples, num_examples=NUM_EXAMPLE_QUERIES)

            del retriever
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"[ERROR] Backbone {backbone_name} failed: {e}")

    if len(all_metrics) > 0:
        df_metrics = pd.DataFrame(all_metrics).sort_values(by="mAP", ascending=False)
        summary_csv = os.path.join(RESULTS_DIR, "retrieval_comparison_summary.csv")
        df_metrics.to_csv(summary_csv, index=False)

        print("\nFinal comparison:")
        print(df_metrics)
        print(f"\nSaved summary to: {summary_csv}")
    else:
        print("No successful backbone runs.")


# if __name__ == "__main__":
#     main()

In [ ]:
import os
from typing import List, Optional
import numpy as np
import pandas as pd


def load_saved_query_and_results(model_dir: str, q_idx: int):
    test_index_csv = os.path.join(model_dir, "test_index.csv")
    test_embed_npy = os.path.join(model_dir, "test_embeddings.npy")
    retrieval_csv = os.path.join(model_dir, "retrieval_examples.csv")

    if not os.path.isfile(test_index_csv):
        raise FileNotFoundError(f"Missing file: {test_index_csv}")
    if not os.path.isfile(test_embed_npy):
        raise FileNotFoundError(f"Missing file: {test_embed_npy}")
    if not os.path.isfile(retrieval_csv):
        raise FileNotFoundError(f"Missing file: {retrieval_csv}")

    test_index_df = pd.read_csv(test_index_csv)
    test_embeddings = np.load(test_embed_npy).astype(np.float32)
    retrieval_df = pd.read_csv(retrieval_csv)

    if q_idx < 0 or q_idx >= len(test_index_df):
        raise IndexError(f"q_idx={q_idx} out of range [0, {len(test_index_df)-1}]")

    query_path = test_index_df.iloc[q_idx]["image_path"]
    query_emb = test_embeddings[q_idx]

    results_for_query = (
        retrieval_df[retrieval_df["query_path"] == query_path]
        .sort_values("rank")
        .to_dict(orient="records")
    )

    if len(results_for_query) == 0:
        raise ValueError(f"No retrieval results found for query_path: {query_path}")

    return query_emb, results_for_query, query_path, test_index_df


def show_pca_with_similarity_coloring(
    query_emb: np.ndarray,
    results: List[dict],
    index_csv: str,
    embed_npy: str,
    query_path: Optional[str] = None,
    top_k: Optional[int] = None,
):
    try:
        import matplotlib.pyplot as plt
        from sklearn.decomposition import PCA
    except Exception:
        print("Please install scikit-learn and matplotlib:")
        print("!pip install -q scikit-learn matplotlib")
        return

    if not os.path.isfile(index_csv):
        raise FileNotFoundError(f"Missing file: {index_csv}")
    if not os.path.isfile(embed_npy):
        raise FileNotFoundError(f"Missing file: {embed_npy}")

    index_df = pd.read_csv(index_csv)
    db_embeddings = np.load(embed_npy).astype(np.float32)
    db_embeddings = db_embeddings / (np.linalg.norm(db_embeddings, axis=1, keepdims=True) + 1e-8)

    query_emb = np.asarray(query_emb, dtype=np.float32)
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-8)

    sims = db_embeddings @ query_emb

    if top_k is not None:
        results = results[:top_k]

    retrieved_paths = []
    for item in results:
        if "image_path" in item:
            retrieved_paths.append(item["image_path"])
        elif "retrieved_path" in item:
            retrieved_paths.append(item["retrieved_path"])
        else:
            raise KeyError("Each result must contain either 'image_path' or 'retrieved_path'.")

    db_paths_abs = [os.path.abspath(p) for p in index_df["image_path"].tolist()]
    path_to_idx = {p: i for i, p in enumerate(db_paths_abs)}

    retrieved_indices = [
        path_to_idx[os.path.abspath(p)]
        for p in retrieved_paths
        if os.path.abspath(p) in path_to_idx
    ]

    query_db_indices = []
    if query_path is not None:
        query_abs = os.path.abspath(query_path)
        query_db_indices = [i for i, p in enumerate(db_paths_abs) if p == query_abs]

    X = np.vstack([db_embeddings, query_emb[None, :]])
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)
    explained = pca.explained_variance_ratio_
    query_idx = len(X) - 1

    all_db_indices = np.arange(len(db_embeddings))
    query_db_set = set(query_db_indices)
    mask = np.array([i not in query_db_set for i in all_db_indices])

    plt.figure(figsize=(11, 8))

    sc = plt.scatter(
        X_2d[:-1][mask, 0],
        X_2d[:-1][mask, 1],
        c=sims[mask],
        s=20,
        alpha=0.55,
    )
    plt.colorbar(sc, label="Cosine similarity to query")

    cmap = sc.cmap
    norm = sc.norm

    for rank_i, db_idx in enumerate(retrieved_indices, start=1):
        x, y = X_2d[db_idx]

        class_txt = ""
        if "class_name" in index_df.columns:
            class_txt = f"\n{index_df.iloc[db_idx]['class_name']}"

        plt.scatter(
            x, y,
            s=180,
            c=[sims[db_idx]],
            cmap=cmap,
            norm=norm,
            edgecolors="black",
            linewidths=1.4,
            zorder=3,
        )
        plt.text(
            x, y,
            f"Top {rank_i}{class_txt}",
            fontsize=9,
            fontweight="bold",
            zorder=4,
        )

    qx, qy = X_2d[query_idx]
    plt.scatter(
        qx, qy,
        s=300,
        marker="*",
        c="red",
        edgecolors="black",
        linewidths=1.4,
        label="Query",
        zorder=5,
    )
    plt.text(qx, qy, "QUERY", fontsize=10, fontweight="bold", zorder=6)

    plt.title(
        f"PCA of 2D image embeddings colored by cosine similarity\n"
        f"PC1: {explained[0]*100:.2f}% variance, PC2: {explained[1]*100:.2f}% variance"
    )
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_saved_query_pca(
    model_dir: str,
    q_idx: int,
    top_k: int = 5,
):
    query_emb, results_for_query, query_path, test_index_df = load_saved_query_and_results(
        model_dir=model_dir,
        q_idx=q_idx,
    )

    show_pca_with_similarity_coloring(
        query_emb=query_emb,
        results=results_for_query,
        index_csv=os.path.join(model_dir, "train_index.csv"),
        embed_npy=os.path.join(model_dir, "train_embeddings.npy"),
        query_path=query_path,
        top_k=top_k,
    )

    print(f"Query index: {q_idx}")
    print(f"Query path: {query_path}")
    if "class_name" in test_index_df.columns:
        print(f"Query class: {test_index_df.iloc[q_idx]['class_name']}")

# model_dir = os.path.join(RESULTS_DIR, "medsiglip")
# plot_saved_query_pca(model_dir=model_dir, q_idx=0, top_k=5)

In [ ]:
import os
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd


def load_saved_query_and_results(model_dir: str, q_idx: int) -> Tuple[np.ndarray, List[dict], str, pd.DataFrame]:
    test_index_csv = os.path.join(model_dir, "test_index.csv")
    test_embed_npy = os.path.join(model_dir, "test_embeddings.npy")
    retrieval_csv = os.path.join(model_dir, "retrieval_examples.csv")

    for path in (test_index_csv, test_embed_npy, retrieval_csv):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Missing file: {path}")

    test_index_df = pd.read_csv(test_index_csv)
    test_embeddings = np.load(test_embed_npy).astype(np.float32)
    retrieval_df = pd.read_csv(retrieval_csv)

    if q_idx < 0 or q_idx >= len(test_index_df):
        raise IndexError(f"q_idx={q_idx} out of range")

    query_path = test_index_df.iloc[q_idx]["image_path"]
    query_emb = test_embeddings[q_idx]

    results_for_query = (
        retrieval_df[retrieval_df["query_path"] == query_path]
        .sort_values("rank")
        .to_dict(orient="records")
    )

    if len(results_for_query) == 0:
        raise ValueError(f"No retrieval results found for query_path: {query_path}")

    return query_emb, results_for_query, query_path, test_index_df


def _extract_retrieved_paths(results: List[dict], top_k: Optional[int] = None) -> List[str]:
    if top_k is not None:
        results = results[:top_k]

    retrieved_paths = []
    for item in results:
        if "image_path" in item:
            retrieved_paths.append(item["image_path"])
        elif "retrieved_path" in item:
            retrieved_paths.append(item["retrieved_path"])
        else:
            raise KeyError("Missing image path key")
    return retrieved_paths


def _build_class_colors(class_names: List[str], cmap_name: str = "tab20") -> dict:
    import matplotlib.pyplot as plt

    unique_classes = sorted(pd.unique(class_names))
    cmap = plt.get_cmap(cmap_name, max(len(unique_classes), 1))
    return {cls_name: cmap(i) for i, cls_name in enumerate(unique_classes)}


def show_pca_with_class_coloring(
    query_emb: np.ndarray,
    results: List[dict],
    index_csv: str,
    embed_npy: str,
    query_path: Optional[str] = None,
    top_k: Optional[int] = None,
):
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA

    if not os.path.isfile(index_csv) or not os.path.isfile(embed_npy):
        raise FileNotFoundError("Missing index or embedding file")

    index_df = pd.read_csv(index_csv)
    db_embeddings = np.load(embed_npy).astype(np.float32)

    if "class_name" not in index_df.columns:
        raise KeyError("index_csv must contain 'class_name'")

    # Normalize
    db_embeddings = db_embeddings / (np.linalg.norm(db_embeddings, axis=1, keepdims=True) + 1e-8)
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-8)

    # Retrieved indices
    retrieved_paths = _extract_retrieved_paths(results, top_k=top_k)
    db_paths_abs = [os.path.abspath(p) for p in index_df["image_path"].tolist()]
    path_to_idx = {p: i for i, p in enumerate(db_paths_abs)}

    retrieved_indices = [
        path_to_idx[os.path.abspath(p)]
        for p in retrieved_paths
        if os.path.abspath(p) in path_to_idx
    ]

    # PCA
    X = np.vstack([db_embeddings, query_emb[None, :]])
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)
    explained = pca.explained_variance_ratio_
    query_idx = len(X) - 1

    # Colors
    class_names = index_df["class_name"].astype(str).tolist()
    class_to_color = _build_class_colors(class_names)

    plt.figure(figsize=(10, 8))

    # Plot all points (smaller size)
    for cls_name in sorted(set(class_names)):
        mask = index_df["class_name"].astype(str).values == cls_name
        plt.scatter(
            X_2d[:-1][mask, 0],
            X_2d[:-1][mask, 1],
            s=18,  # smaller points
            alpha=0.6,
            color=class_to_color[cls_name],
            label=cls_name,
        )

    # Highlight retrieved
    for rank_i, db_idx in enumerate(retrieved_indices, start=1):
        x, y = X_2d[db_idx]
        cls_name = str(index_df.iloc[db_idx]["class_name"])

        plt.scatter(
            x,
            y,
            s=140,
            color=class_to_color[cls_name],
            edgecolors="black",
            linewidths=1.5,
            zorder=4,
        )

        # Small clean labels (ONLY top-3)
        if rank_i <= 3:
            plt.text(
                x,
                y,
                f"{rank_i}",
                fontsize=7,
                ha="center",
                va="center",
                zorder=5,
                bbox=dict(
                    facecolor="white",
                    alpha=0.7,
                    edgecolor="none",
                    boxstyle="round,pad=0.2",
                ),
            )

    # Query point
    qx, qy = X_2d[query_idx]
    plt.scatter(
        qx,
        qy,
        s=250,
        marker="*",
        color="red",
        edgecolors="black",
        linewidths=1.5,
        label="Query",
        zorder=6,
    )

    plt.title(
        f"PCA of embeddings (colored by class)\n"
        f"PC1: {explained[0]*100:.2f}%, PC2: {explained[1]*100:.2f}%"
    )
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, alpha=0.3)

    # Clean legend (avoid clutter)
    if len(set(class_names)) <= 15:
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()


def plot_saved_query_pca_by_class(model_dir: str, q_idx: int, top_k: int = 5):
    query_emb, results, query_path, test_index_df = load_saved_query_and_results(
        model_dir=model_dir,
        q_idx=q_idx,
    )

    show_pca_with_class_coloring(
        query_emb=query_emb,
        results=results,
        index_csv=os.path.join(model_dir, "train_index.csv"),
        embed_npy=os.path.join(model_dir, "train_embeddings.npy"),
        query_path=query_path,
        top_k=top_k,
    )

    print(f"Query index: {q_idx}")
    print(f"Query path: {query_path}")
    if "class_name" in test_index_df.columns:
        print(f"Query class: {test_index_df.iloc[q_idx]['class_name']}")


# Usage
# model_dir = os.path.join(RESULTS_DIR, "resnet50__frozen")
# plot_saved_query_pca_by_class(model_dir=model_dir, q_idx=0, top_k=5)

# Usage
# model_dir = os.path.join(RESULTS_DIR, "resnet50__full_finetune")
# plot_saved_query_pca_by_class(model_dir=model_dir, q_idx=0, top_k=5)

In [ ]:
# ===================================== REVIEWER-ONLY EXTENSIONS (FULL WORKING CELL) =====================================
# This cell adds ONLY what the reviewers asked for:
#   1) Statistical significance analysis
#   2) Fine-tuning experiments
#   3) Systematic error analysis
#
# IMPORTANT:
# - This cell assumes your ORIGINAL benchmark code is already defined above.
# - It FIXES the "Inference tensors cannot be saved for backward" error by using a TRAINABLE
#   encoding path instead of UniversalImageRetriever.encode_images() during fine-tuning.
# - Replace your previous reviewer cell with this one, then run:
#       run_reviewer_extensions_main()
# ======================================================================================================================

import os
import gc
import copy
import numpy as np
import pandas as pd
from typing import List, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


# ===================================== CONFIG =====================================
RUN_STATISTICAL_ANALYSIS = True
RUN_FINE_TUNING = True
RUN_ERROR_ANALYSIS = True

EVAL_MODES = ["frozen", "linear_probe", "metric_finetune", "full_finetune"]

# Fine-tune only these to keep things stable and practical
FINE_TUNE_BACKBONES = [
    "resnet50",
    "vit_b16",
    "swin_b",
    "clip_vit_l14",
    "siglip",
    "openclip_vit_h14",
]

FT_EPOCHS = 8
FT_BATCH_SIZE = 16
FT_LR_HEAD = 1e-3
FT_LR_BACKBONE = 1e-5
FT_WEIGHT_DECAY = 1e-4
FT_LABEL_SMOOTHING = 0.1
FT_UNFREEZE_LAST_N_BLOCKS = 2

N_BOOTSTRAPS = 1000
N_RANDOMIZATION_TRIALS = 5000
SIGNIFICANCE_ALPHA = 0.05
REFERENCE_EXPERIMENT = None   # e.g. "openclip_vit_h14__frozen"; None => best mAP experiment

NUM_HARDEST_EXAMPLES_TO_SAVE = 50

# Assumes these are already defined in your original code:
# DATASET_ROOT, RESULTS_DIR, BACKBONES_TO_TEST, IMAGE_SIZE, TOP_KS, BATCH_SIZE, NUM_WORKERS,
# EMBED_DIM, USE_BFLOAT16, RANDOM_SEED, SHOW_RESULTS, NUM_EXAMPLE_QUERIES


# ===================================== SPLIT =====================================
def stratified_train_val_test_split(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    seed: int = 42,
):
    counts = df["label"].value_counts()
    valid_labels = counts[counts >= 3].index.tolist()
    dropped = df[~df["label"].isin(valid_labels)].copy()
    df = df[df["label"].isin(valid_labels)].copy().reset_index(drop=True)

    if len(dropped) > 0:
        print(f"[WARN] Dropped {len(dropped)} images from classes with <3 images")

    trainval_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"].values,
    )

    val_relative = val_size / (1.0 - test_size)
    train_df, val_df = train_test_split(
        trainval_df,
        test_size=val_relative,
        random_state=seed,
        stratify=trainval_df["label"].values,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


# ===================================== CLASSIFICATION DATASET =====================================
class ImageFolderClassificationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def _preprocess(self, img):
        img = img.resize((self.image_size, self.image_size))
        x = np.asarray(img, dtype=np.float32) / 255.0
        x = np.transpose(x, (2, 0, 1))
        x = torch.from_numpy(x)

        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
        x = (x - mean) / std
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_rgb(row["image_path"])
        x = self._preprocess(img)
        y = int(row["label"])
        return x, y


# ===================================== TRAINABLE RETRIEVER =====================================
# Uses your existing UniversalImageRetriever, but bypasses encode_images() during training
# because encode_images() is decorated with @torch.inference_mode() in your original code.
class TrainableUniversalImageRetriever(UniversalImageRetriever):
    def encode_images_trainable(self, x: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        encoder_params = list(self.encoder.parameters())
        enc_dtype = encoder_params[0].dtype if len(encoder_params) > 0 else torch.float32

        # Input comes ImageNet-normalized from dataset -> adapt for backbone
        x = x.to(device=device, dtype=torch.float32)
        x = self._adapt_input_for_backbone(x)

        # Resize if needed
        if x.shape[-1] != self.image_size or x.shape[-2] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size), mode="bilinear", align_corners=False)

        # For SAM-style backbones, use float32
        if self.encoder_kind in {"sam_hf_slice2d", "sam_repo_slice2d"}:
            x = x.to(device=device, dtype=torch.float32)
        else:
            x = x.to(device=device, dtype=enc_dtype)

        feats = self._encode_2d_batch(x)

        if self.proj is not None:
            feats = self.proj(feats.float())

        feats = F.normalize(feats.float(), p=2, dim=-1)
        return feats


class RetrievalFineTuneModel(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        num_classes: int,
        embed_dim: int = 512,
        use_bfloat16: bool = True,
        image_size: int = 224,
    ):
        super().__init__()
        self.retriever = TrainableUniversalImageRetriever(
            backbone_name=backbone_name,
            embed_dim=embed_dim,
            use_bfloat16=use_bfloat16,
            image_size=image_size,
        )
        self.classifier = nn.Linear(self.retriever.output_dim, num_classes)

    def forward(self, x):
        feats = self.retriever.encode_images_trainable(x)
        logits = self.classifier(feats)
        return logits, feats


def set_trainable_layers(model: RetrievalFineTuneModel, mode: str, unfreeze_last_n_blocks: int = 2):
    for p in model.retriever.encoder.parameters():
        p.requires_grad = False

    if model.retriever.proj is not None:
        for p in model.retriever.proj.parameters():
            p.requires_grad = True

    for p in model.classifier.parameters():
        p.requires_grad = True

    if mode == "linear_probe":
        return

    if mode == "full_finetune":
        for p in model.retriever.encoder.parameters():
            p.requires_grad = True
        return

    if mode == "metric_finetune":
        enc = model.retriever.encoder
        found = False

        # timm ViT
        if hasattr(enc, "blocks") and isinstance(enc.blocks, nn.ModuleList) and len(enc.blocks) > 0:
            for blk in enc.blocks[-unfreeze_last_n_blocks:]:
                for p in blk.parameters():
                    p.requires_grad = True
            found = True

        # Swin-like
        if hasattr(enc, "layers") and isinstance(enc.layers, nn.ModuleList) and len(enc.layers) > 0:
            for blk in enc.layers[-unfreeze_last_n_blocks:]:
                for p in blk.parameters():
                    p.requires_grad = True
            found = True

        # HuggingFace CLIP/SigLIP-like
        if hasattr(enc, "vision_model"):
            vm = enc.vision_model
            if hasattr(vm, "encoder") and hasattr(vm.encoder, "layers"):
                layers = vm.encoder.layers
                if isinstance(layers, nn.ModuleList) and len(layers) > 0:
                    for blk in layers[-unfreeze_last_n_blocks:]:
                        for p in blk.parameters():
                            p.requires_grad = True
                    found = True

        # OpenCLIP visual transformer fallback
        if hasattr(enc, "visual"):
            vis = enc.visual
            if hasattr(vis, "transformer") and hasattr(vis.transformer, "resblocks"):
                rbs = vis.transformer.resblocks
                if isinstance(rbs, nn.ModuleList) and len(rbs) > 0:
                    for blk in rbs[-unfreeze_last_n_blocks:]:
                        for p in blk.parameters():
                            p.requires_grad = True
                    found = True

        if not found:
            for p in model.retriever.encoder.parameters():
                p.requires_grad = True


# ===================================== LOSS =====================================
class SupervisedPrototypeLoss(nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor, labels: torch.Tensor):
        features = F.normalize(features, dim=1)
        unique_labels = labels.unique(sorted=True)

        if len(unique_labels) < 2:
            return torch.tensor(0.0, device=features.device, dtype=features.dtype)

        prototypes = []
        proto_labels = []
        for lab in unique_labels:
            mask = (labels == lab)
            proto = features[mask].mean(dim=0)
            proto = F.normalize(proto, dim=0)
            prototypes.append(proto)
            proto_labels.append(lab)

        prototypes = torch.stack(prototypes, dim=0)
        proto_labels = torch.stack(proto_labels).to(labels.device)

        logits = features @ prototypes.T / self.temperature

        target = []
        for lab in labels:
            idx = (proto_labels == lab).nonzero(as_tuple=True)[0][0]
            target.append(idx)
        target = torch.stack(target)

        return F.cross_entropy(logits, target)


# ===================================== TRAINING =====================================
def train_finetune_model(
    backbone_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    num_classes: int,
    mode: str,
    device: str,
    embed_dim: int = 512,
    use_bfloat16: bool = True,
    image_size: int = 224,
    epochs: int = 8,
):
    model = RetrievalFineTuneModel(
        backbone_name=backbone_name,
        num_classes=num_classes,
        embed_dim=embed_dim,
        use_bfloat16=use_bfloat16,
        image_size=image_size,
    ).to(device)

    set_trainable_layers(model, mode=mode, unfreeze_last_n_blocks=FT_UNFREEZE_LAST_N_BLOCKS)

    train_ds = ImageFolderClassificationDataset(train_df, image_size=image_size)
    val_ds = ImageFolderClassificationDataset(val_df, image_size=image_size)

    train_loader = DataLoader(
        train_ds,
        batch_size=FT_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=FT_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    head_params = list(model.classifier.parameters())
    if model.retriever.proj is not None:
        head_params += list(model.retriever.proj.parameters())
    head_param_ids = {id(p) for p in head_params}

    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_param_ids]

    param_groups = [{"params": head_params, "lr": FT_LR_HEAD, "weight_decay": FT_WEIGHT_DECAY}]
    if len(backbone_params) > 0:
        param_groups.append({"params": backbone_params, "lr": FT_LR_BACKBONE, "weight_decay": FT_WEIGHT_DECAY})

    optimizer = torch.optim.AdamW(param_groups)
    criterion_cls = nn.CrossEntropyLoss(label_smoothing=FT_LABEL_SMOOTHING)
    criterion_proto = SupervisedPrototypeLoss()

    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits, feats = model(x)

            loss_cls = criterion_cls(logits, y)
            if mode == "metric_finetune":
                loss_metric = criterion_proto(feats, y)
                loss = 0.5 * loss_cls + 0.5 * loss_metric
            else:
                loss = loss_cls

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logits, _ = model(x)
                preds = logits.argmax(dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

        train_loss = train_loss / max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_acc = val_correct / max(val_total, 1)

        print(f"[FT][{backbone_name}][{mode}] Epoch {epoch}/{epochs} | loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return model, best_val_acc


@torch.no_grad()
def build_embeddings_from_finetuned_model(
    df: pd.DataFrame,
    model: RetrievalFineTuneModel,
    split_name: str,
    batch_size: int = 32,
    num_workers: int = 2,
    image_size: int = 224,
):
    dataset = ImageFolderRetrievalDataset(df=df, image_size=image_size)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    rows = []
    embeds = []

    model.eval()
    device = next(model.parameters()).device

    for i, batch in enumerate(loader, start=1):
        x = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].cpu().numpy()
        class_names = batch["class_name"]
        image_paths = batch["image_path"]

        feats = model.retriever.encode_images_trainable(x).detach().cpu().numpy().astype(np.float32)
        embeds.append(feats)

        for j in range(len(image_paths)):
            rows.append({
                "image_path": image_paths[j],
                "label": int(labels[j]),
                "class_name": class_names[j],
                "split": split_name,
            })

        if i % 10 == 0 or i == len(loader):
            print(f"[{split_name}] Processed {i}/{len(loader)} batches")

    embeddings = np.vstack(embeds).astype(np.float32)
    index_df = pd.DataFrame(rows)
    return index_df, embeddings


# ===================================== EVALUATION =====================================
def evaluate_retrieval_with_per_query(
    query_df: pd.DataFrame,
    query_embeddings: np.ndarray,
    gallery_df: pd.DataFrame,
    gallery_embeddings: np.ndarray,
    ks=(1, 5, 10),
):
    query_labels = query_df["label"].values
    gallery_labels = gallery_df["label"].values

    max_k = max(ks)
    topk_idx, topk_sims = retrieve_topk(
        query_embeddings=query_embeddings,
        gallery_embeddings=gallery_embeddings,
        top_k=max_k,
    )

    precision_at_k = {k: [] for k in ks}
    hitrate_at_k = {k: [] for k in ks}
    recall_at_k = {k: [] for k in ks}
    ap_list = []
    rr_list = []
    examples = []
    per_query_rows = []

    gallery_label_counts = pd.Series(gallery_labels).value_counts().to_dict()

    for q_idx in range(len(query_df)):
        q_label = int(query_labels[q_idx])
        q_class = query_df.iloc[q_idx]["class_name"]
        q_path = query_df.iloc[q_idx]["image_path"]

        retrieved_indices = topk_idx[q_idx]
        retrieved_sims = topk_sims[q_idx]
        retrieved_labels = gallery_labels[retrieved_indices]
        relevant_flags = (retrieved_labels == q_label).astype(np.int32).tolist()

        total_relevant_in_gallery = int(gallery_label_counts.get(q_label, 0))

        qrow = {
            "query_idx": q_idx,
            "query_path": q_path,
            "query_class": q_class,
            "query_label": q_label,
            "num_relevant_in_gallery": total_relevant_in_gallery,
        }

        for k in ks:
            flags_k = relevant_flags[:k]
            num_rel_k = sum(flags_k)

            p_at_k = num_rel_k / k
            h_at_k = 1.0 if num_rel_k > 0 else 0.0
            r_at_k = num_rel_k / total_relevant_in_gallery if total_relevant_in_gallery > 0 else 0.0

            precision_at_k[k].append(p_at_k)
            hitrate_at_k[k].append(h_at_k)
            recall_at_k[k].append(r_at_k)

            qrow[f"Precision@{k}"] = p_at_k
            qrow[f"HitRate@{k}"] = h_at_k
            qrow[f"Recall@{k}"] = r_at_k

        ap = average_precision_from_flags(relevant_flags)
        rr = reciprocal_rank(relevant_flags)
        ap_list.append(ap)
        rr_list.append(rr)

        qrow["AP"] = ap
        qrow["RR"] = rr
        qrow["top1_correct"] = int(relevant_flags[0]) if len(relevant_flags) > 0 else 0
        qrow["first_correct_rank"] = next((i + 1 for i, flag in enumerate(relevant_flags) if flag), -1)

        per_query_rows.append(qrow)

        ex = {
            "query_path": q_path,
            "query_class": q_class,
            "query_label": q_label,
            "retrieved": []
        }
        for rank, (g_idx, sim) in enumerate(zip(retrieved_indices, retrieved_sims), start=1):
            ex["retrieved"].append({
                "rank": rank,
                "image_path": gallery_df.iloc[g_idx]["image_path"],
                "class_name": gallery_df.iloc[g_idx]["class_name"],
                "label": int(gallery_df.iloc[g_idx]["label"]),
                "similarity": float(sim),
                "correct_class": bool(gallery_df.iloc[g_idx]["label"] == q_label),
            })
        examples.append(ex)

    metrics = {
        "num_queries": int(len(query_df)),
        "gallery_size": int(len(gallery_df)),
        "mAP": float(np.mean(ap_list)) if len(ap_list) > 0 else 0.0,
        "MRR": float(np.mean(rr_list)) if len(rr_list) > 0 else 0.0,
    }

    for k in ks:
        metrics[f"Precision@{k}"] = float(np.mean(precision_at_k[k])) if len(precision_at_k[k]) > 0 else 0.0
        metrics[f"HitRate@{k}"] = float(np.mean(hitrate_at_k[k])) if len(hitrate_at_k[k]) > 0 else 0.0
        metrics[f"Recall@{k}"] = float(np.mean(recall_at_k[k])) if len(recall_at_k[k]) > 0 else 0.0

    per_query_df = pd.DataFrame(per_query_rows)
    return metrics, examples, per_query_df


# ===================================== STATISTICAL ANALYSIS =====================================
def bootstrap_ci(values, n_bootstraps=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=np.float64)
    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan

    samples = []
    for _ in range(n_bootstraps):
        idx = rng.integers(0, n, size=n)
        samples.append(values[idx].mean())

    samples = np.asarray(samples, dtype=np.float64)
    mean_ = float(values.mean())
    lo = float(np.percentile(samples, 100 * (alpha / 2)))
    hi = float(np.percentile(samples, 100 * (1 - alpha / 2)))
    return mean_, lo, hi


def compute_metric_cis(per_query_df: pd.DataFrame, ks=(1, 5, 10), n_bootstraps=1000, alpha=0.05):
    metrics_to_eval = ["AP", "RR"]
    for k in ks:
        metrics_to_eval += [f"Precision@{k}", f"HitRate@{k}", f"Recall@{k}"]

    rows = []
    for m in metrics_to_eval:
        mean_, lo, hi = bootstrap_ci(per_query_df[m].values, n_bootstraps=n_bootstraps, alpha=alpha)
        name = {"AP": "mAP", "RR": "MRR"}.get(m, m)
        rows.append({
            "metric": name,
            "mean": mean_,
            "ci_lower": lo,
            "ci_upper": hi,
        })
    return pd.DataFrame(rows)


def paired_bootstrap_test(values_a, values_b, n_bootstraps=1000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(values_a, dtype=np.float64)
    b = np.asarray(values_b, dtype=np.float64)
    assert len(a) == len(b), "Paired bootstrap needs matched queries"

    observed_diff = float(a.mean() - b.mean())
    n = len(a)
    diffs = []

    for _ in range(n_bootstraps):
        idx = rng.integers(0, n, size=n)
        diffs.append(a[idx].mean() - b[idx].mean())

    diffs = np.asarray(diffs, dtype=np.float64)
    p_value = float(np.mean(np.abs(diffs) >= abs(observed_diff)))
    return observed_diff, p_value


def paired_randomization_test(values_a, values_b, n_trials=5000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(values_a, dtype=np.float64)
    b = np.asarray(values_b, dtype=np.float64)
    assert len(a) == len(b), "Randomization test needs matched queries"

    observed_diff = float(a.mean() - b.mean())
    count = 0

    for _ in range(n_trials):
        swap = rng.integers(0, 2, size=len(a)).astype(bool)
        aa = a.copy()
        bb = b.copy()
        tmp = aa[swap].copy()
        aa[swap] = bb[swap]
        bb[swap] = tmp
        diff = aa.mean() - bb.mean()
        if abs(diff) >= abs(observed_diff):
            count += 1

    p_value = float((count + 1) / (n_trials + 1))
    return observed_diff, p_value


def compare_experiments_statistically(
    all_per_query: Dict[str, pd.DataFrame],
    reference_experiment: Optional[str] = None,
    ks=(1, 5, 10),
):
    if len(all_per_query) < 2:
        return pd.DataFrame()

    if reference_experiment is None:
        best_name = None
        best_map = -1.0
        for name, dfq in all_per_query.items():
            cur = float(dfq["AP"].mean())
            if cur > best_map:
                best_map = cur
                best_name = name
        reference_experiment = best_name

    ref_df = all_per_query[reference_experiment]
    rows = []

    metric_pairs = [("AP", "mAP"), ("RR", "MRR")]
    for k in ks:
        metric_pairs += [
            (f"Precision@{k}", f"Precision@{k}"),
            (f"HitRate@{k}", f"HitRate@{k}"),
            (f"Recall@{k}", f"Recall@{k}"),
        ]

    for name, cmp_df in all_per_query.items():
        if name == reference_experiment:
            continue

        merged = ref_df.merge(cmp_df, on="query_path", suffixes=("_ref", "_cmp"))

        for raw_metric, display_metric in metric_pairs:
            diff_boot, p_boot = paired_bootstrap_test(
                merged[f"{raw_metric}_cmp"].values,
                merged[f"{raw_metric}_ref"].values,
                n_bootstraps=N_BOOTSTRAPS,
                seed=RANDOM_SEED,
            )
            _, p_rand = paired_randomization_test(
                merged[f"{raw_metric}_cmp"].values,
                merged[f"{raw_metric}_ref"].values,
                n_trials=N_RANDOMIZATION_TRIALS,
                seed=RANDOM_SEED,
            )

            rows.append({
                "reference_experiment": reference_experiment,
                "compared_experiment": name,
                "metric": display_metric,
                "mean_difference_cmp_minus_ref": diff_boot,
                "bootstrap_pvalue": p_boot,
                "randomization_pvalue": p_rand,
                "significant_at_0.05": int((p_boot < 0.05) and (p_rand < 0.05)),
            })

    return pd.DataFrame(rows)


# ===================================== ERROR ANALYSIS =====================================
def compute_per_class_metrics(per_query_df: pd.DataFrame, ks=(1, 5, 10)):
    agg = {"AP": "mean", "RR": "mean", "top1_correct": "mean"}
    for k in ks:
        agg[f"Precision@{k}"] = "mean"
        agg[f"HitRate@{k}"] = "mean"
        agg[f"Recall@{k}"] = "mean"

    out = per_query_df.groupby(["query_label", "query_class"], as_index=False).agg(agg)
    out = out.rename(columns={"AP": "mAP", "RR": "MRR", "top1_correct": "Top1Acc"})
    return out.sort_values("mAP", ascending=True).reset_index(drop=True)


def extract_confusion_pairs(examples: List[Dict], top_n: int = 50):
    rows = []
    for ex in examples:
        qclass = ex["query_class"]
        for item in ex["retrieved"]:
            if not item["correct_class"]:
                rows.append({
                    "query_class": qclass,
                    "confused_with": item["class_name"],
                })

    if len(rows) == 0:
        return pd.DataFrame(columns=["query_class", "confused_with", "count"])

    out = pd.DataFrame(rows)
    out = out.value_counts(["query_class", "confused_with"]).reset_index(name="count")
    out = out.sort_values("count", ascending=False).head(top_n).reset_index(drop=True)
    return out


def hardest_queries_table(per_query_df: pd.DataFrame, examples: List[Dict], top_n: int = 50):
    ex_map = {ex["query_path"]: ex for ex in examples}
    df = per_query_df.copy()
    df["difficulty_score"] = (1.0 - df["AP"]) + (1.0 - df["RR"]) + (1.0 - df["top1_correct"])

    rows = []
    for _, row in df.sort_values(["difficulty_score", "AP"], ascending=False).head(top_n).iterrows():
        ex = ex_map[row["query_path"]]
        first_wrong = next((item for item in ex["retrieved"] if not item["correct_class"]), None)

        rows.append({
            "query_path": row["query_path"],
            "query_class": row["query_class"],
            "AP": row["AP"],
            "RR": row["RR"],
            "top1_correct": row["top1_correct"],
            "first_correct_rank": row["first_correct_rank"],
            "first_wrong_class": None if first_wrong is None else first_wrong["class_name"],
            "first_wrong_similarity": None if first_wrong is None else first_wrong["similarity"],
        })

    return pd.DataFrame(rows)


def embedding_similarity_analysis(train_df, train_embeddings, test_df, test_embeddings):
    train_labels = train_df["label"].values
    test_labels = test_df["label"].values

    train_emb = train_embeddings / (np.linalg.norm(train_embeddings, axis=1, keepdims=True) + 1e-8)
    test_emb = test_embeddings / (np.linalg.norm(test_embeddings, axis=1, keepdims=True) + 1e-8)

    sims = test_emb @ train_emb.T
    same_class = (test_labels[:, None] == train_labels[None, :])

    intra = sims[same_class]
    inter = sims[~same_class]

    return pd.DataFrame([{
        "intra_mean": float(np.mean(intra)) if len(intra) else np.nan,
        "intra_std": float(np.std(intra)) if len(intra) else np.nan,
        "inter_mean": float(np.mean(inter)) if len(inter) else np.nan,
        "inter_std": float(np.std(inter)) if len(inter) else np.nan,
        "margin_intra_minus_inter": float(np.mean(intra) - np.mean(inter)) if len(intra) and len(inter) else np.nan,
    }])


# ===================================== MAIN =====================================
def run_reviewer_extensions_main():
    set_seed(RANDOM_SEED)
    os.makedirs(RESULTS_DIR, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    df = list_images_by_class(DATASET_ROOT)
    print(f"Total images found: {len(df)}")
    print(f"Total classes found: {df['label'].nunique()}")

    if RUN_FINE_TUNING:
        train_df, val_df, test_df = stratified_train_val_test_split(
            df=df,
            test_size=0.2,
            val_size=0.1,
            seed=RANDOM_SEED,
        )
        print(f"Train size: {len(train_df)}")
        print(f"Val size:   {len(val_df)}")
        print(f"Test size:  {len(test_df)}")
    else:
        train_df, test_df = stratified_split(df, test_size=0.2, seed=RANDOM_SEED)
        val_df = None
        print(f"Train size: {len(train_df)}")
        print(f"Test size:  {len(test_df)}")

    num_classes = int(df["label"].nunique())

    all_metrics = []
    all_per_query = {}
    all_examples = {}

    modes_to_run = ["frozen"] if not RUN_FINE_TUNING else EVAL_MODES

    for backbone_name in BACKBONES_TO_TEST:
        for mode in modes_to_run:
            if mode != "frozen" and backbone_name not in FINE_TUNE_BACKBONES:
                continue

            experiment_name = f"{backbone_name}__{mode}"
            print("\n" + "=" * 100)
            print(f"EXPERIMENT: {experiment_name}")
            print("=" * 100)

            try:
                if mode == "frozen":
                    retriever = UniversalImageRetriever(
                        backbone_name=backbone_name,
                        embed_dim=EMBED_DIM,
                        use_bfloat16=USE_BFLOAT16,
                        image_size=IMAGE_SIZE,
                    ).to(device).eval()

                    train_index_df, train_embeddings = build_embeddings(
                        df=train_df,
                        retriever=retriever,
                        split_name="train",
                        batch_size=BATCH_SIZE,
                        num_workers=NUM_WORKERS,
                        image_size=IMAGE_SIZE,
                    )

                    test_index_df, test_embeddings = build_embeddings(
                        df=test_df,
                        retriever=retriever,
                        split_name="test",
                        batch_size=BATCH_SIZE,
                        num_workers=NUM_WORKERS,
                        image_size=IMAGE_SIZE,
                    )

                    del retriever

                else:
                    model, best_val_acc = train_finetune_model(
                        backbone_name=backbone_name,
                        train_df=train_df,
                        val_df=val_df,
                        num_classes=num_classes,
                        mode=mode,
                        device=device,
                        embed_dim=EMBED_DIM,
                        use_bfloat16=USE_BFLOAT16,
                        image_size=IMAGE_SIZE,
                        epochs=FT_EPOCHS,
                    )

                    train_index_df, train_embeddings = build_embeddings_from_finetuned_model(
                        df=train_df,
                        model=model,
                        split_name="train",
                        batch_size=BATCH_SIZE,
                        num_workers=NUM_WORKERS,
                        image_size=IMAGE_SIZE,
                    )

                    test_index_df, test_embeddings = build_embeddings_from_finetuned_model(
                        df=test_df,
                        model=model,
                        split_name="test",
                        batch_size=BATCH_SIZE,
                        num_workers=NUM_WORKERS,
                        image_size=IMAGE_SIZE,
                    )

                    del model

                safe_name = experiment_name.replace("/", "_")
                out_dir = os.path.join(RESULTS_DIR, safe_name)
                os.makedirs(out_dir, exist_ok=True)

                save_split_and_embeddings(train_index_df, train_embeddings, "train", out_dir)
                save_split_and_embeddings(test_index_df, test_embeddings, "test", out_dir)

                metrics, examples, per_query_df = evaluate_retrieval_with_per_query(
                    query_df=test_index_df,
                    query_embeddings=test_embeddings,
                    gallery_df=train_index_df,
                    gallery_embeddings=train_embeddings,
                    ks=TOP_KS,
                )

                metrics["backbone"] = backbone_name
                metrics["mode"] = mode
                metrics["experiment"] = experiment_name
                if mode != "frozen":
                    metrics["best_val_acc"] = float(best_val_acc)

                all_metrics.append(metrics)
                all_per_query[experiment_name] = per_query_df.copy()
                all_examples[experiment_name] = examples

                print(pd.DataFrame([metrics]).T)

                per_query_df.to_csv(os.path.join(out_dir, "per_query_metrics.csv"), index=False)

                retrieval_rows = []
                for ex in examples:
                    for item in ex["retrieved"]:
                        retrieval_rows.append({
                            "query_path": ex["query_path"],
                            "query_class": ex["query_class"],
                            "rank": item["rank"],
                            "retrieved_path": item["image_path"],
                            "retrieved_class": item["class_name"],
                            "similarity": item["similarity"],
                            "correct_class": item["correct_class"],
                        })
                pd.DataFrame(retrieval_rows).to_csv(os.path.join(out_dir, "retrieval_examples.csv"), index=False)

                if RUN_STATISTICAL_ANALYSIS:
                    ci_df = compute_metric_cis(
                        per_query_df=per_query_df,
                        ks=TOP_KS,
                        n_bootstraps=N_BOOTSTRAPS,
                        alpha=SIGNIFICANCE_ALPHA,
                    )
                    ci_df.to_csv(os.path.join(out_dir, "confidence_intervals.csv"), index=False)

                if RUN_ERROR_ANALYSIS:
                    per_class_df = compute_per_class_metrics(per_query_df, ks=TOP_KS)
                    per_class_df.to_csv(os.path.join(out_dir, "per_class_metrics.csv"), index=False)

                    confusion_df = extract_confusion_pairs(examples, top_n=50)
                    confusion_df.to_csv(os.path.join(out_dir, "confusion_pairs.csv"), index=False)

                    hardest_df = hardest_queries_table(
                        per_query_df=per_query_df,
                        examples=examples,
                        top_n=NUM_HARDEST_EXAMPLES_TO_SAVE,
                    )
                    hardest_df.to_csv(os.path.join(out_dir, "hardest_queries.csv"), index=False)

                    sim_df = embedding_similarity_analysis(
                        train_df=train_index_df,
                        train_embeddings=train_embeddings,
                        test_df=test_index_df,
                        test_embeddings=test_embeddings,
                    )
                    sim_df.to_csv(os.path.join(out_dir, "embedding_similarity_stats.csv"), index=False)

                if SHOW_RESULTS:
                    print(f"Showing example retrievals for {experiment_name}")
                    show_example_retrievals(examples, num_examples=NUM_EXAMPLE_QUERIES)

                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"[ERROR] Experiment {experiment_name} failed: {e}")
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    if len(all_metrics) == 0:
        print("No successful reviewer-focused runs.")
        return

    df_metrics = pd.DataFrame(all_metrics).sort_values(by="mAP", ascending=False).reset_index(drop=True)
    df_metrics.to_csv(os.path.join(RESULTS_DIR, "retrieval_comparison_summary.csv"), index=False)
    print("\nFinal comparison:")
    print(df_metrics)

    if RUN_STATISTICAL_ANALYSIS and len(all_per_query) >= 2:
        sig_df = compare_experiments_statistically(
            all_per_query=all_per_query,
            reference_experiment=REFERENCE_EXPERIMENT,
            ks=TOP_KS,
        )
        sig_df.to_csv(os.path.join(RESULTS_DIR, "statistical_significance_vs_reference.csv"), index=False)
        print("\nSaved statistical significance table.")

    print(f"\nAll reviewer-focused outputs saved to: {RESULTS_DIR}")


# Run:
# run_reviewer_extensions_main()

In [ ]:
# !zip -r retrieval_outputs_2d.zip retrieval_outputs_2d/

In [ ]:
#!cp -r retrieval_outputs_2d.zip /content/drive/MyDrive/Lung_Colon/

In [ ]:
# ===================================== 2D IMAGE RETRIEVAL BENCHMARK =====================================
# Folder structure:
#   dataset_root/
#       class_a/
#           image1.jpg
#           image2.png
#       class_b/
#           image3.jpg
#           ...
#
# What it does:
#   - Optional stratified subset selection (e.g. 5% of data)
#   - Stratified 80/20 split by class
#   - Extract embeddings for both train and test splits
#   - Retrieval: for each test image, retrieve top-k most similar train images
#   - Evaluate whether retrieved neighbors have the same class as the query
#   - Compare computational time across backbones
#   - Estimate runtime for full 100% dataset from 5% subset
#
# Metrics:
#   - Precision@K
#   - HitRate@K
#   - Recall@K
#   - mAP
#   - MRR
#
# Added models:
#   - MedSigLIP
#   - SigLIP
#   - UNI (medical foundation model; requires access if gated)
#   - OpenCLIP ViT-H/14
# ================================================================================================

import os
import gc
import time
import math
import random
from pathlib import Path
from typing import List, Dict, Optional, Tuple

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Install if needed:
# !pip install -q timm transformers open_clip_torch segment-anything huggingface_hub pillow scikit-learn

import timm
from sklearn.model_selection import train_test_split
from transformers import CLIPVisionModel, SamModel, AutoModel
from huggingface_hub import hf_hub_download

try:
    import open_clip
    HAS_OPEN_CLIP = True
except Exception:
    HAS_OPEN_CLIP = False

try:
    from segment_anything import sam_model_registry
    HAS_SEGMENT_ANYTHING = True
except Exception:
    HAS_SEGMENT_ANYTHING = False


# ===================================== USER CONFIG =====================================
DATASET_ROOT = "/content/WCE"
RESULTS_DIR = "./retrieval_outputs_2d"
os.makedirs(RESULTS_DIR, exist_ok=True)

BACKBONES_TO_TEST = [
    "resnet50",
    "vit_b16",
    "swin_b",
    "clip_vit_l14",
    "biomedclip",
    "dinov2_vitb14",
    "mae_vit_b16",
    "bmc_clip_cf",
    "sam_vit_b",
    "medsam_vit_b",
    "siglip",
    "medsiglip",
    "uni",
    "openclip_vit_h14",
]

MODEL_PATHS = {
    "clip_vit_l14": "openai/clip-vit-large-patch14",
    "biomedclip": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    "sam_vit_b": "facebook/sam-vit-base",

    # Put your local MedSAM checkpoint here
    "medsam_vit_b_ckpt": "/content/ViT/medsam_vit_b.pth",

    # BMC-CLIP-CF
    "bmc_clip_cf_repo": "BIOMEDICA/BMC_CLIP_CF",
    "bmc_clip_cf_file": "BMC_CLIP_CF.pt",
    "bmc_clip_cf_base_model": "ViT-L-14",
    "bmc_clip_cf_base_pretrained": "commonpool_xl_clip_s13b_b90k",
    "bmc_clip_cf_alpha": 0.0,

    # Added
    "siglip": "google/siglip-base-patch16-224",
    "medsiglip": "google/medsiglip-448",

    # UNI can be gated; ensure access if needed
    "uni": "hf-hub:MahmoodLab/UNI",

    # OpenCLIP ViT-H/14
    "openclip_vit_h14_model": "ViT-H-14",
    "openclip_vit_h14_pretrained": "laion2b_s32b_b79k",
}

IMAGE_SIZE = 224
TOP_KS = (1, 5, 10)
BATCH_SIZE = 32
NUM_WORKERS = 2
EMBED_DIM = 512
USE_BFLOAT16 = True
RANDOM_SEED = 42
SHOW_RESULTS = True
NUM_EXAMPLE_QUERIES = 5

ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# ===================================== SUBSET / TIMING CONFIG =====================================
USE_SUBSET_FOR_TIMING = True
SUBSET_FRACTION = 0.05           # 5%
MIN_IMAGES_PER_CLASS_IN_SUBSET = 2
PRINT_ESTIMATED_FULL_DATASET_MESSAGES = True
SAVE_TIMING_SUMMARY = True
# ================================================================================================


# ===================================== NORMALIZATION =====================================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

CLIP_MEAN = torch.tensor([0.48145466, 0.45782750, 0.40821073], dtype=torch.float32).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], dtype=torch.float32).view(1, 3, 1, 1)


def imagenet_norm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(device=x.device, dtype=x.dtype)
    std = IMAGENET_STD.to(device=x.device, dtype=x.dtype)
    return (x - mean) / std


def clip_norm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = CLIP_MEAN.to(device=x.device, dtype=x.dtype)
    std = CLIP_STD.to(device=x.device, dtype=x.dtype)
    return (x - mean) / std


def imagenet_unnorm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(device=x.device, dtype=x.dtype)
    std = IMAGENET_STD.to(device=x.device, dtype=x.dtype)
    return x * std + mean


# ===================================== UTILS =====================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_open_rgb(path: str) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        raise RuntimeError(f"Could not open image {path}: {e}")


def list_images_by_class(root: str) -> pd.DataFrame:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {root}")

    rows = []
    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])

    if len(class_dirs) == 0:
        raise ValueError(f"No class folders found in {root}")

    for class_dir in class_dirs:
        class_name = class_dir.name
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in ALLOWED_EXTS:
                rows.append({
                    "image_path": str(p),
                    "class_name": class_name,
                })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise ValueError(f"No images found in {root}")

    classes = sorted(df["class_name"].unique().tolist())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    df["label"] = df["class_name"].map(class_to_idx)
    return df


def stratified_split(df: pd.DataFrame, test_size: float = 0.2, seed: int = 42):
    counts = df["label"].value_counts()
    valid_labels = counts[counts >= 2].index.tolist()
    dropped = df[~df["label"].isin(valid_labels)].copy()
    df = df[df["label"].isin(valid_labels)].copy().reset_index(drop=True)

    if len(dropped) > 0:
        print(f"[WARN] Dropped {len(dropped)} images from classes with <2 images")

    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"].values,
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def stratified_subsample_df(
    df: pd.DataFrame,
    fraction: float = 0.05,
    seed: int = 42,
    min_images_per_class: int = 2,
) -> pd.DataFrame:
    """
    Create a stratified subset from the full dataframe.
    Keeps at least min_images_per_class per class if possible.
    """
    if not (0 < fraction <= 1.0):
        raise ValueError(f"fraction must be in (0,1], got {fraction}")

    if fraction >= 1.0:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    rng = np.random.RandomState(seed)
    parts = []

    for label, group in df.groupby("label", sort=False):
        n = len(group)
        if n < min_images_per_class:
            continue

        k = max(min_images_per_class, int(round(n * fraction)))
        k = min(k, n)
        idx = rng.choice(group.index.values, size=k, replace=False)
        parts.append(df.loc[idx])

    if len(parts) == 0:
        raise ValueError("Subset selection produced no data.")

    sub_df = pd.concat(parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return sub_df


def average_precision_from_flags(relevant_flags: List[int]) -> float:
    relevant_flags = np.asarray(relevant_flags, dtype=np.int32)
    total_rel = relevant_flags.sum()
    if total_rel == 0:
        return 0.0

    precisions = []
    hits = 0
    for i, rel in enumerate(relevant_flags, start=1):
        if rel:
            hits += 1
            precisions.append(hits / i)
    return float(np.mean(precisions)) if precisions else 0.0


def reciprocal_rank(relevant_flags: List[int]) -> float:
    for i, rel in enumerate(relevant_flags, start=1):
        if rel:
            return 1.0 / i
    return 0.0


def format_seconds(sec: float) -> str:
    if sec < 60:
        return f"{sec:.2f} sec"
    if sec < 3600:
        return f"{sec / 60:.2f} min"
    return f"{sec / 3600:.2f} hr"


def estimate_full_dataset_times(
    subset_total_images: int,
    full_total_images: int,
    subset_train_size: int,
    subset_test_size: int,
    full_train_size: int,
    full_test_size: int,
    train_embed_time: float,
    test_embed_time: float,
    retrieval_eval_time: float,
) -> Dict[str, float]:
    """
    Embedding extraction ~ O(N)
    Retrieval/eval ~ O(N_test * N_train)
    """
    if subset_total_images <= 0 or subset_train_size <= 0 or subset_test_size <= 0:
        raise ValueError("Subset sizes must be positive")

    embed_scale = full_total_images / subset_total_images
    retrieval_scale = (full_test_size * full_train_size) / (subset_test_size * subset_train_size)

    est_train_embed = train_embed_time * embed_scale
    est_test_embed = test_embed_time * embed_scale
    est_retrieval_eval = retrieval_eval_time * retrieval_scale
    est_total = est_train_embed + est_test_embed + est_retrieval_eval

    return {
        "embed_scale": float(embed_scale),
        "retrieval_scale": float(retrieval_scale),
        "est_train_embed_time_sec_100pct": float(est_train_embed),
        "est_test_embed_time_sec_100pct": float(est_test_embed),
        "est_retrieval_eval_time_sec_100pct": float(est_retrieval_eval),
        "est_total_time_sec_100pct": float(est_total),
    }


def print_full_dataset_estimate_message(
    backbone_name: str,
    subset_fraction: float,
    subset_total_images: int,
    full_total_images: int,
    subset_train_size: int,
    subset_test_size: int,
    full_train_size: int,
    full_test_size: int,
    timing_info: Dict[str, float],
):
    print("\n" + "-" * 100)
    print(f"[ESTIMATION MESSAGE] Backbone: {backbone_name}")
    print("-" * 100)
    print(
        f"Measured runtime on {subset_fraction * 100:.1f}% subset "
        f"({subset_total_images}/{full_total_images} images)."
    )
    print(
        f"Subset split: train={subset_train_size}, test={subset_test_size} | "
        f"Full split estimate: train={full_train_size}, test={full_test_size}"
    )
    print(
        f"Assumption: embedding time scales ~ linearly with number of images, "
        f"retrieval/evaluation scales ~ with (test_size × train_size)."
    )
    print(f"Estimated full 100% train embedding time: {format_seconds(timing_info['est_train_embed_time_sec_100pct'])}")
    print(f"Estimated full 100% test embedding time:  {format_seconds(timing_info['est_test_embed_time_sec_100pct'])}")
    print(f"Estimated full 100% retrieval+eval time: {format_seconds(timing_info['est_retrieval_eval_time_sec_100pct'])}")
    print(f"Estimated full 100% total runtime:       {format_seconds(timing_info['est_total_time_sec_100pct'])}")
    print("-" * 100 + "\n")


# ===================================== DATASET =====================================
class ImageFolderRetrievalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def _preprocess(self, img: Image.Image) -> torch.Tensor:
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        x = np.asarray(img, dtype=np.float32) / 255.0
        x = np.transpose(x, (2, 0, 1))
        x = torch.from_numpy(x)

        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
        x = (x - mean) / std
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_rgb(row["image_path"])
        x = self._preprocess(img)
        return {
            "image": x,
            "label": int(row["label"]),
            "class_name": row["class_name"],
            "image_path": row["image_path"],
        }


# ===================================== BMC-CLIP-CF LOADER =====================================
def load_bmc_clip_cf():
    if not HAS_OPEN_CLIP:
        raise ImportError("BMC_CLIP_CF requires open_clip_torch.")

    repo_id = MODEL_PATHS["bmc_clip_cf_repo"]
    filename = MODEL_PATHS["bmc_clip_cf_file"]
    model_name = MODEL_PATHS["bmc_clip_cf_base_model"]
    pretrained = MODEL_PATHS["bmc_clip_cf_base_pretrained"]
    alpha = float(MODEL_PATHS.get("bmc_clip_cf_alpha", 0.0))

    ckpt_path = hf_hub_download(repo_id=repo_id, filename=filename)
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint["state_dict"]
    del checkpoint

    model, _, _ = open_clip.create_model_and_transforms(
        model_name=model_name,
        pretrained=pretrained,
    )

    model_state_dict = model.state_dict()

    print(f"Merging BMC-CLIP-CF with alpha={alpha}")
    if alpha == 0:
        print("Using only BMC-CLIP-CF checkpoint weights")

    for key in model_state_dict.keys():
        ckpt_key = f"module.{key}"
        if ckpt_key in state_dict:
            model_state_dict[key] = alpha * model_state_dict[key] + (1 - alpha) * state_dict[ckpt_key]

    model.load_state_dict(model_state_dict, strict=True)
    return model


# ===================================== FEATURE EXTRACTORS =====================================
def create_feature_extractor(backbone_name: str):
    """
    Returns:
      model, feat_dim, encoder_kind, preferred_image_size
    """
    backbone_name = backbone_name.lower()

    if backbone_name == "resnet50":
        m = timm.create_model("resnet50", pretrained=True, num_classes=0)
        return m, m.num_features, "timm_slice2d", 224

    elif backbone_name == "vit_b16":
        m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
        return m, m.embed_dim, "timm_slice2d", 224

    elif backbone_name == "swin_b":
        m = timm.create_model("swin_base_patch4_window7_224", pretrained=True, num_classes=0)
        return m, m.num_features, "timm_slice2d", 224

    elif backbone_name == "clip_vit_l14":
        m = CLIPVisionModel.from_pretrained(MODEL_PATHS["clip_vit_l14"])
        return m, m.config.hidden_size, "clip_slice2d", 224

    elif backbone_name == "biomedclip":
        if not HAS_OPEN_CLIP:
            raise ImportError("BiomedCLIP requires open_clip_torch.")
        m, _, _ = open_clip.create_model_and_transforms(MODEL_PATHS["biomedclip"])
        feat_dim = getattr(m.visual, "output_dim", 512)
        return m, feat_dim, "openclip_slice2d", 224

    elif backbone_name == "dinov2_vitb14":
        m = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        return m, m.embed_dim, "dinov2_slice2d", 224

    elif backbone_name == "mae_vit_b16":
        m = timm.create_model("vit_base_patch16_224.mae", pretrained=True, num_classes=0)
        return m, m.embed_dim, "timm_slice2d", 224

    elif backbone_name == "bmc_clip_cf":
        m = load_bmc_clip_cf()
        feat_dim = getattr(m.visual, "output_dim", 768)
        return m, feat_dim, "openclip_slice2d", 224

    elif backbone_name == "sam_vit_b":
        m = SamModel.from_pretrained(MODEL_PATHS["sam_vit_b"])
        feat_dim = int(getattr(m.config.vision_config, "output_channels", 256))
        return m, feat_dim, "sam_hf_slice2d", 1024

    elif backbone_name == "medsam_vit_b":
        if not HAS_SEGMENT_ANYTHING:
            raise ImportError("MedSAM requires segment-anything.")
        ckpt = MODEL_PATHS["medsam_vit_b_ckpt"]
        if not os.path.isfile(ckpt):
            raise FileNotFoundError(f"MedSAM checkpoint not found: {ckpt}")
        m = sam_model_registry["vit_b"](checkpoint=ckpt)
        return m, 256, "sam_repo_slice2d", 512

    elif backbone_name == "siglip":
        m = AutoModel.from_pretrained(MODEL_PATHS["siglip"])
        hidden = getattr(getattr(m, "config", None), "hidden_size", None)
        if hidden is None and hasattr(m, "vision_model") and hasattr(m.vision_model.config, "hidden_size"):
            hidden = m.vision_model.config.hidden_size
        if hidden is None:
            hidden = 768
        return m, hidden, "siglip_slice2d", 224

    elif backbone_name == "medsiglip":
        m = AutoModel.from_pretrained(MODEL_PATHS["medsiglip"])
        hidden = getattr(getattr(m, "config", None), "hidden_size", None)
        if hidden is None and hasattr(m, "vision_model") and hasattr(m.vision_model.config, "hidden_size"):
            hidden = m.vision_model.config.hidden_size
        if hidden is None:
            hidden = 768
        return m, hidden, "medsiglip_slice2d", 448

    elif backbone_name == "uni":
        try:
            m = timm.create_model(MODEL_PATHS["uni"], pretrained=True, num_classes=0)
        except Exception:
            m = timm.create_model(
                MODEL_PATHS["uni"],
                pretrained=True,
                num_classes=0,
                init_values=1e-5,
                dynamic_img_size=True,
            )

        feat_dim = getattr(m, "num_features", None)
        if feat_dim is None:
            feat_dim = getattr(m, "embed_dim", 1024)
        return m, feat_dim, "uni_slice2d", 224

    elif backbone_name == "openclip_vit_h14":
        if not HAS_OPEN_CLIP:
            raise ImportError("openclip_vit_h14 requires open_clip_torch.")
        m, _, _ = open_clip.create_model_and_transforms(
            model_name=MODEL_PATHS["openclip_vit_h14_model"],
            pretrained=MODEL_PATHS["openclip_vit_h14_pretrained"],
        )
        feat_dim = getattr(m.visual, "output_dim", 1024)
        return m, feat_dim, "openclip_slice2d", 224

    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")


# ===================================== RETRIEVER =====================================
class UniversalImageRetriever(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        embed_dim: Optional[int] = 512,
        use_bfloat16: bool = True,
        image_size: int = 224,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.encoder, self.hidden_size, self.encoder_kind, self.preferred_image_size = create_feature_extractor(self.backbone_name)

        self.image_size = self.preferred_image_size if self.preferred_image_size is not None else image_size

        self.proj = None
        self.output_dim = self.hidden_size
        if embed_dim is not None and embed_dim != self.hidden_size:
            self.proj = nn.Linear(self.hidden_size, embed_dim)
            self.output_dim = embed_dim

        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()

        safe_bf16_kinds = {
            "timm_slice2d",
            "clip_slice2d",
            "openclip_slice2d",
            "dinov2_slice2d",
            "uni_slice2d",
            "siglip_slice2d",
            "medsiglip_slice2d",
        }
        if use_bfloat16 and torch.cuda.is_available() and self.encoder_kind in safe_bf16_kinds:
            self.encoder = self.encoder.to(dtype=torch.bfloat16)

    def _adapt_input_for_backbone(self, x: torch.Tensor) -> torch.Tensor:
        x = imagenet_unnorm_batch(x.float()).clamp(0.0, 1.0)

        if self.encoder_kind in {"clip_slice2d", "openclip_slice2d", "siglip_slice2d", "medsiglip_slice2d"}:
            x = clip_norm_batch(x)
        elif self.encoder_kind in {"timm_slice2d", "dinov2_slice2d", "sam_hf_slice2d", "sam_repo_slice2d", "uni_slice2d"}:
            x = imagenet_norm_batch(x)
        else:
            x = imagenet_norm_batch(x)

        return x

    def _normalize_encoder_output(self, feats):
        if isinstance(feats, dict):
            for key in [
                "image_embeds",
                "x_norm_clstoken",
                "x_cls_token",
                "cls_token",
                "pooler_output",
                "last_hidden_state",
                "x_norm_patchtokens",
            ]:
                if key in feats:
                    feats = feats[key]
                    break
            else:
                first_key = list(feats.keys())[0]
                feats = feats[first_key]

        if isinstance(feats, (list, tuple)):
            feats = feats[0]

        if hasattr(feats, "last_hidden_state"):
            feats = feats.last_hidden_state

        if not torch.is_tensor(feats):
            raise TypeError(f"Unsupported feature output type: {type(feats)}")

        if feats.ndim == 4:
            feats = feats.mean(dim=(2, 3))
        elif feats.ndim == 3:
            feats = feats.mean(dim=1)
        elif feats.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected feature tensor shape: {tuple(feats.shape)}")

        return feats

    def _forward_in_chunks(self, x: torch.Tensor, fn, chunk_size: int = 2) -> torch.Tensor:
        outs = []
        for i in range(0, x.shape[0], chunk_size):
            xi = x[i:i + chunk_size]
            yi = fn(xi)
            outs.append(yi)
        return torch.cat(outs, dim=0)

    def _encode_2d_batch(self, x2d: torch.Tensor) -> torch.Tensor:
        if self.encoder_kind == "timm_slice2d":
            if self.backbone_name == "swin_b":
                feats = self.encoder.forward_features(x2d)

                if isinstance(feats, (list, tuple)):
                    feats = feats[0]

                if not torch.is_tensor(feats):
                    raise TypeError(f"Unsupported Swin output type: {type(feats)}")

                if feats.ndim == 4:
                    if feats.shape[-1] == self.hidden_size:
                        feats = feats.mean(dim=(1, 2))
                    elif feats.shape[1] == self.hidden_size:
                        feats = feats.mean(dim=(2, 3))
                    else:
                        raise ValueError(f"Unexpected Swin feature shape: {tuple(feats.shape)}")
                elif feats.ndim == 3:
                    feats = feats.mean(dim=1)
                elif feats.ndim == 2:
                    pass
                else:
                    raise ValueError(f"Unexpected Swin feature shape: {tuple(feats.shape)}")

                return feats

            else:
                if hasattr(self.encoder, "forward_features"):
                    feats = self.encoder.forward_features(x2d)
                else:
                    feats = self.encoder(x2d)
                feats = self._normalize_encoder_output(feats)
                return feats

        elif self.encoder_kind == "clip_slice2d":
            out = self.encoder(pixel_values=x2d)
            feats = out.last_hidden_state.mean(dim=1)
            return feats

        elif self.encoder_kind == "openclip_slice2d":
            if hasattr(self.encoder, "encode_image"):
                feats = self.encoder.encode_image(x2d)
            else:
                feats = self.encoder.visual(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "dinov2_slice2d":
            if hasattr(self.encoder, "forward_features"):
                feats = self.encoder.forward_features(x2d)
            else:
                feats = self.encoder(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "sam_hf_slice2d":
            x = x2d.float()
            if x.max() <= 1.0:
                x = x * 255.0

            feats = self._forward_in_chunks(
                x,
                lambda z: self.encoder.get_image_embeddings(pixel_values=z),
                chunk_size=1,
            )
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "sam_repo_slice2d":
            x = x2d.float()
            if x.max() <= 1.0:
                x = x * 255.0

            def run_repo_sam(z):
                zz = self.encoder.preprocess(z) if hasattr(self.encoder, "preprocess") else z
                return self.encoder.image_encoder(zz)

            feats = self._forward_in_chunks(x, run_repo_sam, chunk_size=1)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "siglip_slice2d":
            if hasattr(self.encoder, "get_image_features"):
                feats = self.encoder.get_image_features(pixel_values=x2d)
            else:
                out = self.encoder(pixel_values=x2d)
                if hasattr(out, "image_embeds") and out.image_embeds is not None:
                    feats = out.image_embeds
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state.mean(dim=1)
                else:
                    feats = self._normalize_encoder_output(out)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "medsiglip_slice2d":
            if hasattr(self.encoder, "get_image_features"):
                feats = self.encoder.get_image_features(pixel_values=x2d)
            else:
                out = self.encoder(pixel_values=x2d)
                if hasattr(out, "image_embeds") and out.image_embeds is not None:
                    feats = out.image_embeds
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state.mean(dim=1)
                else:
                    feats = self._normalize_encoder_output(out)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "uni_slice2d":
            if hasattr(self.encoder, "forward_features"):
                feats = self.encoder.forward_features(x2d)
            else:
                feats = self.encoder(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        else:
            raise ValueError(f"Unknown encoder kind: {self.encoder_kind}")

    @torch.inference_mode()
    def encode_images(self, x: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device
        encoder_params = list(self.encoder.parameters())
        enc_dtype = encoder_params[0].dtype if len(encoder_params) > 0 else torch.float32

        x = x.to(device=device, dtype=torch.float32)
        x = self._adapt_input_for_backbone(x)

        if x.shape[-1] != self.image_size or x.shape[-2] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size), mode="bilinear", align_corners=False)

        x = x.to(device=device, dtype=enc_dtype)
        feats = self._encode_2d_batch(x)

        if self.proj is not None:
            feats = self.proj(feats.float())

        feats = F.normalize(feats.float(), p=2, dim=-1)
        return feats


# ===================================== EMBEDDINGS =====================================
def build_embeddings(
    df: pd.DataFrame,
    retriever: UniversalImageRetriever,
    split_name: str,
    batch_size: int = 32,
    num_workers: int = 2,
    image_size: int = 224,
):
    dataset = ImageFolderRetrievalDataset(df=df, image_size=image_size)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    rows = []
    embeds = []

    retriever.eval()

    start_time = time.perf_counter()

    for i, batch in enumerate(loader, start=1):
        x = batch["image"]
        labels = batch["label"].numpy()
        class_names = batch["class_name"]
        image_paths = batch["image_path"]

        feats = retriever.encode_images(x).detach().cpu().numpy().astype(np.float32)
        embeds.append(feats)

        for j in range(len(image_paths)):
            rows.append({
                "image_path": image_paths[j],
                "label": int(labels[j]),
                "class_name": class_names[j],
                "split": split_name,
            })

        if i % 10 == 0 or i == len(loader):
            print(f"[{split_name}] Processed {i}/{len(loader)} batches")

    elapsed = time.perf_counter() - start_time

    embeddings = np.vstack(embeds).astype(np.float32)
    index_df = pd.DataFrame(rows)
    return index_df, embeddings, elapsed


# ===================================== RETRIEVAL =====================================
def retrieve_topk(
    query_embeddings: np.ndarray,
    gallery_embeddings: np.ndarray,
    top_k: int = 5,
):
    query_embeddings = query_embeddings / (np.linalg.norm(query_embeddings, axis=1, keepdims=True) + 1e-8)
    gallery_embeddings = gallery_embeddings / (np.linalg.norm(gallery_embeddings, axis=1, keepdims=True) + 1e-8)

    sims = query_embeddings @ gallery_embeddings.T
    topk_idx = np.argsort(sims, axis=1)[:, ::-1][:, :top_k]
    topk_sims = np.take_along_axis(sims, topk_idx, axis=1)
    return topk_idx, topk_sims


def evaluate_retrieval(
    query_df: pd.DataFrame,
    query_embeddings: np.ndarray,
    gallery_df: pd.DataFrame,
    gallery_embeddings: np.ndarray,
    ks=(1, 5, 10),
):
    eval_start = time.perf_counter()

    query_labels = query_df["label"].values
    gallery_labels = gallery_df["label"].values

    max_k = max(ks)
    topk_idx, topk_sims = retrieve_topk(
        query_embeddings=query_embeddings,
        gallery_embeddings=gallery_embeddings,
        top_k=max_k,
    )

    precision_at_k = {k: [] for k in ks}
    hitrate_at_k = {k: [] for k in ks}
    recall_at_k = {k: [] for k in ks}
    ap_list = []
    rr_list = []
    examples = []

    gallery_label_counts = pd.Series(gallery_labels).value_counts().to_dict()

    for q_idx in range(len(query_df)):
        q_label = int(query_labels[q_idx])
        q_class = query_df.iloc[q_idx]["class_name"]
        q_path = query_df.iloc[q_idx]["image_path"]

        retrieved_indices = topk_idx[q_idx]
        retrieved_sims = topk_sims[q_idx]
        retrieved_labels = gallery_labels[retrieved_indices]
        relevant_flags = (retrieved_labels == q_label).astype(np.int32).tolist()

        total_relevant_in_gallery = int(gallery_label_counts.get(q_label, 0))

        for k in ks:
            flags_k = relevant_flags[:k]
            num_rel_k = sum(flags_k)

            precision_at_k[k].append(num_rel_k / k)
            hitrate_at_k[k].append(1.0 if num_rel_k > 0 else 0.0)

            if total_relevant_in_gallery > 0:
                recall_at_k[k].append(num_rel_k / total_relevant_in_gallery)
            else:
                recall_at_k[k].append(0.0)

        ap_list.append(average_precision_from_flags(relevant_flags))
        rr_list.append(reciprocal_rank(relevant_flags))

        ex = {
            "query_path": q_path,
            "query_class": q_class,
            "retrieved": []
        }
        for rank, (g_idx, sim) in enumerate(zip(retrieved_indices, retrieved_sims), start=1):
            ex["retrieved"].append({
                "rank": rank,
                "image_path": gallery_df.iloc[g_idx]["image_path"],
                "class_name": gallery_df.iloc[g_idx]["class_name"],
                "label": int(gallery_df.iloc[g_idx]["label"]),
                "similarity": float(sim),
                "correct_class": bool(gallery_df.iloc[g_idx]["label"] == q_label),
            })
        examples.append(ex)

    eval_elapsed = time.perf_counter() - eval_start

    metrics = {
        "num_queries": int(len(query_df)),
        "gallery_size": int(len(gallery_df)),
        "mAP": float(np.mean(ap_list)) if len(ap_list) > 0 else 0.0,
        "MRR": float(np.mean(rr_list)) if len(rr_list) > 0 else 0.0,
    }

    for k in ks:
        metrics[f"Precision@{k}"] = float(np.mean(precision_at_k[k])) if len(precision_at_k[k]) > 0 else 0.0
        metrics[f"HitRate@{k}"] = float(np.mean(hitrate_at_k[k])) if len(hitrate_at_k[k]) > 0 else 0.0
        metrics[f"Recall@{k}"] = float(np.mean(recall_at_k[k])) if len(recall_at_k[k]) > 0 else 0.0

    return metrics, examples, eval_elapsed


# ===================================== VIS =====================================
def show_example_retrievals(examples: List[Dict], num_examples: int = 5):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("matplotlib not available, skipping visualization.")
        return

    chosen = examples[:num_examples]
    if len(chosen) == 0:
        return

    for ex in chosen:
        retrieved = ex["retrieved"]
        ncols = len(retrieved) + 1

        plt.figure(figsize=(3 * ncols, 3))

        q_img = safe_open_rgb(ex["query_path"])
        ax = plt.subplot(1, ncols, 1)
        ax.imshow(q_img)
        ax.set_title(f"QUERY\n{ex['query_class']}")
        ax.axis("off")

        for i, item in enumerate(retrieved, start=2):
            img = safe_open_rgb(item["image_path"])
            ax = plt.subplot(1, ncols, i)
            ax.imshow(img)
            ax.set_title(
                f"R{item['rank']}\n{item['class_name']}\nsim={item['similarity']:.3f}\n"
                f"{'OK' if item['correct_class'] else 'WRONG'}",
                fontsize=9
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()


# ===================================== SAVE =====================================
def save_split_and_embeddings(split_df: pd.DataFrame, embeddings: np.ndarray, prefix: str, out_dir: str):
    csv_path = os.path.join(out_dir, f"{prefix}_index.csv")
    npy_path = os.path.join(out_dir, f"{prefix}_embeddings.npy")

    split_df.to_csv(csv_path, index=False)
    np.save(npy_path, embeddings)

    print(f"Saved {prefix} CSV: {csv_path}")
    print(f"Saved {prefix} embeddings: {npy_path}")
    print(f"{prefix} embedding shape: {embeddings.shape}")


# ===================================== MAIN =====================================
def main():
    set_seed(RANDOM_SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    full_df = list_images_by_class(DATASET_ROOT)
    full_total_images = len(full_df)
    full_num_classes = full_df["label"].nunique()

    print(f"Total images found in FULL dataset: {full_total_images}")
    print(f"Total classes found in FULL dataset: {full_num_classes}")
    print("\nFull class distribution:")
    print(full_df["class_name"].value_counts())

    if USE_SUBSET_FOR_TIMING:
        df = stratified_subsample_df(
            full_df,
            fraction=SUBSET_FRACTION,
            seed=RANDOM_SEED,
            min_images_per_class=MIN_IMAGES_PER_CLASS_IN_SUBSET,
        )
        print("\n" + "=" * 100)
        print(f"Using stratified subset for timing/benchmark: {SUBSET_FRACTION * 100:.1f}%")
        print(f"Subset images: {len(df)} / {full_total_images}")
        print(f"Subset classes: {df['label'].nunique()} / {full_num_classes}")
        print("=" * 100)
    else:
        df = full_df.copy()

    print("\nSubset/current class distribution:")
    print(df["class_name"].value_counts())

    train_df, test_df = stratified_split(df, test_size=0.2, seed=RANDOM_SEED)
    print(f"\nCurrent train size: {len(train_df)}")
    print(f"Current test size:  {len(test_df)}")

    full_train_df, full_test_df = stratified_split(full_df, test_size=0.2, seed=RANDOM_SEED)
    full_train_size = len(full_train_df)
    full_test_size = len(full_test_df)

    all_metrics = []
    all_timing_rows = []

    for backbone_name in BACKBONES_TO_TEST:
        print("\n" + "=" * 100)
        print(f"BACKBONE: {backbone_name}")
        print("=" * 100)

        try:
            backbone_start = time.perf_counter()

            retriever = UniversalImageRetriever(
                backbone_name=backbone_name,
                embed_dim=EMBED_DIM,
                use_bfloat16=USE_BFLOAT16,
                image_size=IMAGE_SIZE,
            ).to(device).eval()

            train_index_df, train_embeddings, train_embed_time = build_embeddings(
                df=train_df,
                retriever=retriever,
                split_name="train",
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                image_size=IMAGE_SIZE,
            )

            test_index_df, test_embeddings, test_embed_time = build_embeddings(
                df=test_df,
                retriever=retriever,
                split_name="test",
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                image_size=IMAGE_SIZE,
            )

            safe_name = backbone_name.replace("/", "_")
            out_dir = os.path.join(RESULTS_DIR, safe_name)
            os.makedirs(out_dir, exist_ok=True)

            save_split_and_embeddings(train_index_df, train_embeddings, "train", out_dir)
            save_split_and_embeddings(test_index_df, test_embeddings, "test", out_dir)

            metrics, examples, retrieval_eval_time = evaluate_retrieval(
                query_df=test_index_df,
                query_embeddings=test_embeddings,
                gallery_df=train_index_df,
                gallery_embeddings=train_embeddings,
                ks=TOP_KS,
            )

            total_runtime = time.perf_counter() - backbone_start

            metrics["backbone"] = backbone_name
            metrics["train_embed_time_sec"] = float(train_embed_time)
            metrics["test_embed_time_sec"] = float(test_embed_time)
            metrics["retrieval_eval_time_sec"] = float(retrieval_eval_time)
            metrics["total_runtime_sec"] = float(total_runtime)

            if USE_SUBSET_FOR_TIMING:
                est = estimate_full_dataset_times(
                    subset_total_images=len(df),
                    full_total_images=full_total_images,
                    subset_train_size=len(train_df),
                    subset_test_size=len(test_df),
                    full_train_size=full_train_size,
                    full_test_size=full_test_size,
                    train_embed_time=train_embed_time,
                    test_embed_time=test_embed_time,
                    retrieval_eval_time=retrieval_eval_time,
                )
                metrics.update(est)

                if PRINT_ESTIMATED_FULL_DATASET_MESSAGES:
                    print_full_dataset_estimate_message(
                        backbone_name=backbone_name,
                        subset_fraction=SUBSET_FRACTION,
                        subset_total_images=len(df),
                        full_total_images=full_total_images,
                        subset_train_size=len(train_df),
                        subset_test_size=len(test_df),
                        full_train_size=full_train_size,
                        full_test_size=full_test_size,
                        timing_info=est,
                    )

            all_metrics.append(metrics)

            print("\nMetrics:")
            print(pd.DataFrame([metrics]).T)

            rows = []
            for ex in examples:
                for item in ex["retrieved"]:
                    rows.append({
                        "query_path": ex["query_path"],
                        "query_class": ex["query_class"],
                        "rank": item["rank"],
                        "retrieved_path": item["image_path"],
                        "retrieved_class": item["class_name"],
                        "similarity": item["similarity"],
                        "correct_class": item["correct_class"],
                    })
            pd.DataFrame(rows).to_csv(os.path.join(out_dir, "retrieval_examples.csv"), index=False)

            timing_row = {
                "backbone": backbone_name,
                "subset_fraction_used": SUBSET_FRACTION if USE_SUBSET_FOR_TIMING else 1.0,
                "subset_total_images": len(df),
                "subset_train_size": len(train_df),
                "subset_test_size": len(test_df),
                "full_total_images": full_total_images,
                "full_train_size": full_train_size,
                "full_test_size": full_test_size,
                "train_embed_time_sec": float(train_embed_time),
                "test_embed_time_sec": float(test_embed_time),
                "retrieval_eval_time_sec": float(retrieval_eval_time),
                "total_runtime_sec": float(total_runtime),
            }

            if USE_SUBSET_FOR_TIMING:
                timing_row.update({
                    "estimated_train_embed_time_sec_100pct": metrics["est_train_embed_time_sec_100pct"],
                    "estimated_test_embed_time_sec_100pct": metrics["est_test_embed_time_sec_100pct"],
                    "estimated_retrieval_eval_time_sec_100pct": metrics["est_retrieval_eval_time_sec_100pct"],
                    "estimated_total_time_sec_100pct": metrics["est_total_time_sec_100pct"],
                    "embed_scale": metrics["embed_scale"],
                    "retrieval_scale": metrics["retrieval_scale"],
                })

            all_timing_rows.append(timing_row)

            print("\nTiming summary:")
            print(f"Train embedding time:   {format_seconds(train_embed_time)}")
            print(f"Test embedding time:    {format_seconds(test_embed_time)}")
            print(f"Retrieval + eval time:  {format_seconds(retrieval_eval_time)}")
            print(f"Total runtime:          {format_seconds(total_runtime)}")

            if USE_SUBSET_FOR_TIMING:
                print(f"Estimated 100% total:   {format_seconds(metrics['est_total_time_sec_100pct'])}")

            if SHOW_RESULTS:
                print(f"\nShowing example retrievals for {backbone_name}")
                show_example_retrievals(examples, num_examples=NUM_EXAMPLE_QUERIES)

            del retriever
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"[ERROR] Backbone {backbone_name} failed: {e}")

    if len(all_metrics) > 0:
        df_metrics = pd.DataFrame(all_metrics).sort_values(by="mAP", ascending=False)
        summary_csv = os.path.join(RESULTS_DIR, "retrieval_comparison_summary.csv")
        df_metrics.to_csv(summary_csv, index=False)

        print("\nFinal comparison:")
        print(df_metrics)
        print(f"\nSaved summary to: {summary_csv}")

        if SAVE_TIMING_SUMMARY and len(all_timing_rows) > 0:
            df_timing = pd.DataFrame(all_timing_rows)

            sort_col = "estimated_total_time_sec_100pct" if USE_SUBSET_FOR_TIMING else "total_runtime_sec"
            if sort_col in df_timing.columns:
                df_timing = df_timing.sort_values(by=sort_col, ascending=True)

            timing_csv = os.path.join(RESULTS_DIR, "retrieval_timing_summary.csv")
            df_timing.to_csv(timing_csv, index=False)

            print("\nComputational time comparison:")
            print(df_timing)
            print(f"\nSaved timing summary to: {timing_csv}")

            msg_txt = os.path.join(RESULTS_DIR, "estimated_full_dataset_messages.txt")
            with open(msg_txt, "w", encoding="utf-8") as f:
                for _, row in df_timing.iterrows():
                    f.write("=" * 100 + "\n")
                    f.write(f"Backbone: {row['backbone']}\n")
                    f.write(
                        f"Measured on subset fraction: "
                        f"{float(row['subset_fraction_used']) * 100:.1f}% "
                        f"({int(row['subset_total_images'])}/{int(row['full_total_images'])} images)\n"
                    )
                    f.write(
                        f"Subset split train/test: {int(row['subset_train_size'])}/{int(row['subset_test_size'])}\n"
                    )
                    f.write(
                        f"Full split estimate train/test: {int(row['full_train_size'])}/{int(row['full_test_size'])}\n"
                    )
                    f.write(f"Measured train embedding time: {format_seconds(float(row['train_embed_time_sec']))}\n")
                    f.write(f"Measured test embedding time: {format_seconds(float(row['test_embed_time_sec']))}\n")
                    f.write(f"Measured retrieval+eval time: {format_seconds(float(row['retrieval_eval_time_sec']))}\n")
                    f.write(f"Measured total runtime: {format_seconds(float(row['total_runtime_sec']))}\n")

                    if USE_SUBSET_FOR_TIMING:
                        f.write(
                            "Estimated full 100% runtime assumes: "
                            "embedding scales linearly, retrieval scales with train_size x test_size.\n"
                        )
                        f.write(
                            f"Estimated full train embedding time: "
                            f"{format_seconds(float(row['estimated_train_embed_time_sec_100pct']))}\n"
                        )
                        f.write(
                            f"Estimated full test embedding time: "
                            f"{format_seconds(float(row['estimated_test_embed_time_sec_100pct']))}\n"
                        )
                        f.write(
                            f"Estimated full retrieval+eval time: "
                            f"{format_seconds(float(row['estimated_retrieval_eval_time_sec_100pct']))}\n"
                        )
                        f.write(
                            f"Estimated full total runtime: "
                            f"{format_seconds(float(row['estimated_total_time_sec_100pct']))}\n"
                        )

                    f.write("=" * 100 + "\n\n")

            print(f"Saved estimation messages to: {msg_txt}")

    else:
        print("No successful backbone runs.")


if __name__ == "__main__":
    main()